# Stage 13I — Gemini specialist

Test one four-class head on unassigned passages already predicted as Gemini.
Replay Stage 13F and preserve every decision outside the fixed Gemini gate. Reused development prompts only.


In [ ]:
from pathlib import Path
import hashlib
import subprocess
import sys

SOURCES = {'stage13i_gemini_audit.py': ('"""Test one frozen four-class Gemini specialist on the Stage 13F control."""\nfrom __future__ import annotations\n\nimport argparse\nfrom fractions import Fraction\nimport hashlib\nimport json\nfrom pathlib import Path\nimport time\n\nimport numpy as np\nfrom scipy import __version__ as scipy_version\nfrom sklearn import __version__ as sklearn_version\nfrom threadpoolctl import threadpool_limits\n\nimport stage13b_balancing_stress as reference\nimport stage13c_deberta_complement as stage13c\nimport stage13f_assignment_audit as stage13f\nimport stage13f_assignment_core as assignment\nimport stage13h_residual_core as residual\nimport stage13h_residual_audit as stage13h\nimport stage13i_gemini_core as specialist\n\n\nCOMPLETE = "COMPLETE_STAGE13I_GEMINI_SPECIALIST"\nFAILED = "FAILED_STAGE13I_GEMINI_SPECIALIST"\nSCOPE = "gemini_specialist_on_reused_prompt_heldout_glm_deberta_proxy"\nREPORT_NAME = "gemini_audit_report.json"\nSTAGE13F_CONTROL_CM_SHA = stage13h.STAGE13F_CONTROL_CM_SHA\nSTAGE13F_CONTROL_BA = stage13h.STAGE13F_CONTROL_BA\nSTAGE13H_REPORT_COMMIT = "8b45ab4d2ea32061891f8ca7a2199af51d026d54"\nSTAGE13H_ANCHOR_VERSION = 368859699\nGEMINI = ("gemini-3.1-pro-preview", "gemini-3.6-flash", "gemini-3.7-flash", "gemini-3.8-flash")\n# Nonzero rows of Stage 13H\'s frozen unassigned confusion, restricted to the\n# four Gemini prediction columns, in canonical class order. Counts only.\nELIGIBLE_CM_ROWS = ((2, 1, 0, 0, 0), (5, 47, 5, 2, 2), (6, 11, 31, 8, 5),\n                    (7, 6, 12, 18, 18), (8, 3, 11, 8, 32), (12, 0, 0, 0, 1),\n                    (15, 0, 1, 0, 0), (17, 0, 0, 0, 1))\nEXPECTED_ELIGIBLE = dict(eligible_rows=223, eligible_true_gemini_rows=219,\n                         eligible_true_non_gemini_rows=4, eligible_baseline_errors=95,\n                         eligible_within_family_baseline_errors=91, eligible_baseline_correct=128)\nrequire = reference.require\n\n\ndef replay_residual_gate(evidence, cluster_ids, folds, prompts, y, classes):\n    """Replay the full Stage 13F control and Stage 13H gated cohort before fit."""\n    _, baseline, covered, _, _, cm, digest = residual._replay(\n        evidence, cluster_ids, folds, prompts, y, STAGE13F_CONTROL_CM_SHA)\n    require(digest == STAGE13F_CONTROL_CM_SHA and int(np.trace(cm)) == stage13h.CONTROL_CORRECT\n            and abs(float(reference.recalls(cm).mean()) - STAGE13F_CONTROL_BA) < 1e-15,\n            "stage13i_stage13f_control_replay")\n    labels = tuple(classes)\n    require(tuple(labels[i] for i in range(5, 9)) == GEMINI, "stage13i_gemini_class_axis")\n    family = np.arange(5, 9)\n    eligible = ~covered & np.isin(baseline, family)\n    gated_cm = reference.confusion(y[eligible], baseline[eligible])\n    expected = np.zeros((reference.K, reference.K), dtype=np.int64)\n    for i, *counts in ELIGIBLE_CM_ROWS:\n        expected[i, 5:9] = counts\n    require(int((~covered).sum()) == stage13h.RESIDUAL_ROWS\n            and np.array_equal(gated_cm, expected), "stage13i_stage13h_eligible_confusion_replay")\n    return cm, gated_cm, family\n\n\ndef verify_comparison(comparison, replayed, gated, family):\n    """Verify paired reconciliation and immutable eligible anchor independently."""\n    require(all(comparison[k] == value for k, value in EXPECTED_ELIGIBLE.items()),\n            "stage13i_eligible_counts_replay")\n    require(all(comparison[k] is True for k in (\n        "outside_eligible_unchanged", "predicted_family_unchanged", "fixed_features_and_log_evidence_unchanged",\n        "gate_uses_only_inferred_assignment_and_baseline_prediction")) and comparison["row_level_outputs"] is False,\n        "stage13i_fixed_scope_invariants")\n    baseline = np.asarray(comparison["baseline_confusion_counts"])\n    candidate = np.asarray(comparison["candidate_confusion_counts"])\n    old = np.asarray(comparison["eligible_baseline_confusion_counts"])\n    new = np.asarray(comparison["eligible_candidate_confusion_counts"])\n    shape = (reference.K, reference.K)\n    require(all(x.shape == shape and x.dtype.kind in "iu" and (x >= 0).all()\n                for x in (baseline, candidate, old, new)), "stage13i_comparison_matrix_schema")\n    require(np.array_equal(baseline, replayed) and np.array_equal(old, gated)\n            and np.array_equal(candidate - baseline, new - old)\n            and np.array_equal(old.sum(axis=1), new.sum(axis=1)), "stage13i_paired_matrix_reconciliation")\n    outside = np.setdiff1d(np.arange(reference.K), family)\n    require(not old[:, outside].any() and not new[:, outside].any()\n            and int(new[outside].sum()) == EXPECTED_ELIGIBLE["eligible_true_non_gemini_rows"]\n            and int(np.trace(new)) == comparison["eligible_candidate_correct"], "stage13i_family_gate")\n    def family_matrix(cm):\n        axes = (outside, family)\n        return [[int(cm[np.ix_(i, j)].sum()) for j in axes] for i in axes]\n\n    require(family_matrix(baseline) == family_matrix(candidate)\n            == comparison["baseline_family_confusion_counts"] == comparison["candidate_family_confusion_counts"],\n            "stage13i_family_confusion_reconciliation")\n    supports = baseline.sum(axis=1)\n    require(np.array_equal(supports, candidate.sum(axis=1)) and (supports > 0).all(),\n            "stage13i_class_supports")\n    class_deltas = [Fraction(100 * int(candidate[i, i] - baseline[i, i]), int(supports[i]))\n                    for i in range(reference.K)]\n    exact_delta = sum(class_deltas) / reference.K\n    delta = Fraction(comparison["delta_pp_exact"])\n    require(delta == exact_delta\n            and abs(comparison["baseline_ba"] - STAGE13F_CONTROL_BA) < 1e-15\n            and abs(comparison["candidate_ba"] - float(reference.recalls(candidate).mean())) < 1e-15\n            and abs(comparison["delta_pp"] - float(exact_delta)) < 1e-15,\n            "stage13i_balanced_accuracy_reconciliation")\n    correct_delta = int(np.trace(candidate) - np.trace(baseline))\n    require(correct_delta == comparison["corrected"] - comparison["broken"]\n            and comparison["corrected"] + comparison["broken"] <= comparison["changed"]\n            <= EXPECTED_ELIGIBLE["eligible_rows"],\n            "stage13i_changed_decision_reconciliation")\n    fold_reports = comparison["folds"]\n    require([f["fold"] for f in fold_reports] == list(range(1, reference.FOLDS + 1))\n            and all(f["fit_class_counts"] == [120] * 4 and f["fit_rows"] == 480\n                    and f["fit_prompts"] == 120 and f["heldout_prompts"] == 30 and f["prompt_overlap"] == 0\n                    for f in fold_reports)\n            and all(sum(f[k] for f in fold_reports) == comparison[k]\n                    for k in ("eligible_rows", "corrected", "broken", "changed")),\n            "stage13i_fold_fit_and_pair_coverage")\n    low, high = comparison["conditional_delta_ci95_pp"]\n    require(np.isfinite([low, high]).all() and low <= high, "stage13i_ci_schema")\n    gates = dict(gain_at_least_half_pp=exact_delta >= Fraction(1, 2), ci_lower_positive=low > 0,\n                 no_class_loss_over_five_pp=min(class_deltas) >= -5)\n    require(comparison["passes_development_screen"] is all(gates.values()),\n            "stage13i_development_gate_reconciliation")\n    return gates\n\n\ndef run(input_root, output_root):\n    started = time.monotonic()\n    root, out = Path(input_root), Path(output_root)\n    report = dict(schema_version=1, status=FAILED, scope=SCOPE, fit_attempted=False,\n                  fit_performed=False, cached_probabilities_loaded=False,\n                  stage13c_confusion_replayed=False, stage13f_geometry_replayed=False,\n                  stage13f_control_confusion_replayed=False, stage13h_eligible_confusion_replayed=False,\n                  specialist_fit_attempted=False, specialist_fit_performed=False,\n                  new_variant_scored=False, full_s07_oof_admitted=False, official_score_estimated=False,\n                  official_submission_created=False, competition_test_read=False,\n                  raw_rows_or_predictions_exported=False, uses_reused_development_prompts=True,\n                  s07_typography_guard_used=False, control_assignment_sizes=stage13h.CONTROL_SIZES,\n                  stage13f_anchor_version=stage13h.STAGE13F_ANCHOR_VERSION,\n                  stage13h_anchor_version=STAGE13H_ANCHOR_VERSION,\n                  stage13h_anchor_commit=STAGE13H_REPORT_COMMIT,\n                  gate="Unassigned under Stage 13F and its frozen prediction belongs to Gemini",\n                  specialist_labels=list(GEMINI), specialist_head=stage13c.HEAD,\n                  expected_stage13f_control_confusion_sha256=STAGE13F_CONTROL_CM_SHA,\n                  expected_stage13f_geometry_sha256=stage13h.STAGE13F_GEOMETRY_SHA,\n                  train_sha256=stage13f.TRAIN_SHA, train_ordered_text_sha256=stage13f.TEXT_SHA,\n                  canonical_prompt_fold_mapping_sha256=reference.FOLD_SHA,\n                  expected_input_version_ids=[stage13f.TRAIN_VERSION, reference.SOURCE_VERSION,\n                                              stage13c.DEBERTA_VERSION],\n                  expected_heldout_cache_sha256=reference.CACHE_SHA,\n                  expected_deberta_cache_sha256=stage13c.DEBERTA_SHA,\n                  sklearn_version=sklearn_version, numpy_version=np.__version__, scipy_version=scipy_version)\n    writable = False\n    try:\n        require(not out.exists() or (out.is_dir() and not any(out.iterdir())), "fresh_output_directory")\n        out.mkdir(parents=True, exist_ok=True)\n        writable = True\n        require(sklearn_version == stage13c.SKLEARN_VERSION, "stage13i_sklearn_version")\n        ids, prompts, texts, y, folds, classes, masks = stage13f.train_inputs(reference.one(root, "train.csv"))\n        with threadpool_limits(limits=stage13c.THREADS):\n            geometry, cluster_ids = assignment.geometry(texts, folds, prompts, y)\n        geometry_sha = stage13h.verify_geometry(geometry, cluster_ids, folds)\n        report.update(geometry=geometry, rows=reference.ROWS, prompts=reference.PROMPTS,\n                      classes=classes.tolist(), stage13f_geometry_replayed=True,\n                      stage13f_geometry_sha256=geometry_sha, previously_untouched_rows=stage13h.RESIDUAL_ROWS)\n\n        saved_ids, saved_prompts, saved_y, saved_folds, saved_classes, glm, producer = reference.cache_inputs(root)\n        require(all(np.array_equal(a, b) for a, b in zip((ids, prompts, y, folds, classes),\n                                                        (saved_ids, saved_prompts, saved_y, saved_folds, saved_classes))),\n                "stage13i_heldout_cache_row_or_fold_alignment")\n        report["cached_probabilities_loaded"] = True\n        stage13c.replay_control(glm, y, masks, producer)\n        report["glm_controls_replayed"] = True\n        features, id_mode = stage13c.load_features(reference.one(root, "enc_debv3l2_train.npz"), ids)\n        report["fit_attempted"] = True\n        deb, fit_details, pools = stage13c.fit_oof(features, y, masks)\n        report.update(fit_performed=True, deberta_fit_folds=fit_details, thread_pools=pools,\n                      deberta_id_verification=id_mode, feature_layers=[24, 6])\n        evidence, anchor_digest, residuals = stage13f.replay_stage13c(glm, deb, y, masks)\n        report.update(stage13c_confusion_replayed=True, stage13c_confusion_sha256=anchor_digest,\n                      mixture_weights={"glm": 0.75, "deberta": 0.25},\n                      balancing={"alpha": 1, "iterations": 100, "fold_local": True,\n                                 "maximum_column_mass_errors": residuals})\n        evidence_sha = hashlib.sha256(evidence.astype("<f8", copy=False).tobytes()).hexdigest()\n        baseline, gated, family = replay_residual_gate(evidence, cluster_ids, folds, prompts, y, classes)\n        report.update(stage13f_control_confusion_replayed=True,\n                      stage13f_control_confusion_sha256=STAGE13F_CONTROL_CM_SHA,\n                      stage13h_eligible_confusion_replayed=True, specialist_fit_attempted=True)\n        comparison = specialist.compare(features, evidence, cluster_ids, folds, prompts, y, classes,\n                                        expected_baseline_confusion_sha256=STAGE13F_CONTROL_CM_SHA)\n        gates = verify_comparison(comparison, baseline, gated, family)\n        require(hashlib.sha256(evidence.astype("<f8", copy=False).tobytes()).hexdigest() == evidence_sha,\n                "stage13i_evidence_mutated")\n        require(hashlib.sha256(glm.astype("<f8", copy=False).tobytes()).hexdigest() == reference.PROB_SHA,\n                "stage13i_glm_cache_mutated")\n        report.update(status=COMPLETE, specialist_fit_performed=True, new_variant_scored=True,\n                      comparison=comparison, decision_gates=gates,\n                      specialist_screen_passed=all(gates.values()),\n                      specialist_decision="ELIGIBLE_FOR_SEPARATE_FOLLOW_UP_ONLY" if all(gates.values())\n                      else "KEEP_FROZEN_CONTROL_PAIRED_GATE_FAILED")\n    except reference.GateError as error:\n        report.update(failure_code=str(error))\n    except Exception as error:\n        report.update(failure_code="runtime_failure", failure_type=type(error).__name__)\n    report["seconds"] = round(time.monotonic() - started, 6)\n    if writable:\n        try:\n            saved = out / REPORT_NAME\n            tmp = out / ("." + REPORT_NAME + ".tmp")\n            tmp.write_text(json.dumps(report, sort_keys=True, indent=2, allow_nan=False) + "\\n", encoding="utf-8")\n            tmp.replace(saved)\n            require(json.loads(saved.read_text(encoding="utf-8")) == report, "stage13i_saved_report_mismatch")\n        except Exception as error:\n            report.update(status=FAILED, failure_code="report_write_or_readback_failure", failure_type=type(error).__name__)\n    return report\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--input-root", default="/kaggle/input")\n    parser.add_argument("--output-root", default="/kaggle/working/stage13i_gemini_audit")\n    args = parser.parse_args()\n    result = run(args.input_root, args.output_root)\n    print("STAGE13I_GEMINI_AUDIT_JSON " + json.dumps(result, sort_keys=True, allow_nan=False), flush=True)\n    return 0 if result["status"] == COMPLETE else 2\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'd23c37bb3b35dd36c7de61490aa398f3a0303c655b34c714dd2b707f35e76f54'), 'stage13i_gemini_core.py': ('"""Test one fold-local four-way Gemini readout behind a fixed family gate."""\nfrom __future__ import annotations\n\nfrom fractions import Fraction\nimport warnings\n\nimport numpy as np\nfrom sklearn.exceptions import ConvergenceWarning\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.preprocessing import StandardScaler\nfrom threadpoolctl import threadpool_limits\n\nimport stage13b_balancing_stress as reference\nimport stage13c_deberta_complement as deberta\nimport stage13f_assignment_core as assignment\nimport stage13h_residual_core as residual\n\n\nGEMINI_CLASSES = ("gemini-3.1-pro-preview", "gemini-3.6-flash", "gemini-3.7-flash", "gemini-3.8-flash")\nK = reference.K\nrequire = reference.require\n\n\ndef eligible_gate(covered, baseline, gemini):\n    """Only the existing assignment coverage and baseline prediction gate scoring."""\n    return ~covered & np.isin(baseline, gemini)\n\n\ndef _fit_and_predict(train_x, train_y, score_x, gemini):\n    require(np.array_equal(np.unique(train_y), gemini), "stage13i_specialist_fit_class_axis")\n    scaler = StandardScaler().fit(train_x)\n    with warnings.catch_warnings():\n        warnings.simplefilter("error", ConvergenceWarning)\n        model = LogisticRegression(**deberta.HEAD).fit(scaler.transform(train_x), train_y)\n    require(np.array_equal(model.classes_, gemini), "stage13i_specialist_model_class_axis")\n    require(len(model.n_iter_) == 1 and 0 < model.n_iter_[0] < deberta.HEAD["max_iter"],\n            "stage13i_specialist_iteration_limit")\n    if len(score_x):\n        probabilities = model.predict_proba(scaler.transform(score_x))\n        require(probabilities.shape == (len(score_x), len(gemini))\n                and np.isfinite(probabilities).all() and ((probabilities >= 0) & (probabilities <= 1)).all()\n                and np.allclose(probabilities.sum(axis=1), 1, atol=1e-12, rtol=0), "stage13i_specialist_probabilities")\n        prediction = gemini[probabilities.argmax(axis=1)]\n    else:\n        prediction = np.empty(0, dtype=np.int64)\n    return prediction, int(model.n_iter_[0])\n\n\ndef _conditional_draws(prompts, y, baseline, candidate, rng):\n    unique, prompt_axis = np.unique(prompts, return_inverse=True)\n    n = len(unique)\n    support = np.zeros((n, K), dtype=np.int64)\n    difference = np.zeros_like(support)\n    np.add.at(support, (prompt_axis, y), 1)\n    np.add.at(difference, (prompt_axis, y), (candidate == y).astype(np.int64) - (baseline == y).astype(np.int64))\n    weights = rng.multinomial(n, np.full(n, 1 / n), size=reference.DRAWS)\n    return weights @ support, weights @ difference\n\n\ndef _family_matrix(y, predictions, gemini):\n    return np.bincount(2 * np.isin(y, gemini) + np.isin(predictions, gemini), minlength=4).reshape(2, 2)\n\n\ndef compare(features, evidence, cluster_ids, folds, prompts, y, classes, *, expected_baseline_confusion_sha256):\n    """Compare one fixed specialist with an exactly replayed Stage 13F control."""\n    labels = np.asarray(classes)\n    require(labels.shape == (K,) and labels.dtype.kind == "U"\n            and reference.digest_json(labels.tolist()) == reference.LABEL_SHA, "stage13i_class_axis")\n    gemini = np.flatnonzero(np.isin(labels, GEMINI_CLASSES))\n    require(tuple(labels[gemini]) == GEMINI_CLASSES, "stage13i_gemini_class_axis")\n    require(features.shape == (len(folds), 2048) and features.dtype == np.float64\n            and np.isfinite(features).all(), "stage13i_feature_axis")\n    score_sha = residual._digest(evidence)\n    feature_sha = residual._digest(features)\n    _, baseline, covered, _, _, cm0, digest = residual._replay(\n        evidence, cluster_ids, folds, prompts, y, expected_baseline_confusion_sha256)\n    eligible = eligible_gate(covered, baseline, gemini)\n    require(eligible.any() and not np.any(eligible & covered), "stage13i_no_valid_gate_rows")\n    candidate = baseline.copy()\n    y = y.astype(np.int64, copy=False)\n    true_gemini = np.isin(y, gemini)\n    rows_scored = np.zeros(len(y), dtype=bool)\n    fold_reports = []\n    rng = np.random.default_rng(reference.SEED)\n    support_draws = np.zeros((reference.DRAWS, K), dtype=np.int64)\n    difference_draws = np.zeros_like(support_draws)\n    with threadpool_limits(limits=deberta.THREADS):\n        for fold in np.unique(folds):\n            hold = folds == fold\n            train = ~hold & true_gemini\n            score = hold & eligible\n            fit_prompts = np.unique(prompts[train])\n            held_prompts = np.unique(prompts[hold])\n            require(np.intersect1d(fit_prompts, held_prompts).size == 0\n                    and not rows_scored[score].any(), "stage13i_fold_isolation")\n            predictions, iterations = _fit_and_predict(features[train], y[train], features[score], gemini)\n            require(len(predictions) == int(score.sum()) and np.isin(predictions, gemini).all(),\n                    "stage13i_specialist_gate_predictions")\n            candidate[score], rows_scored[score] = predictions, True\n            yt, p, q = y[hold], baseline[hold], candidate[hold]\n            fcm0, fcm1 = reference.confusion(yt, p), reference.confusion(yt, q)\n            b0, b1 = float(reference.recalls(fcm0).mean()), float(reference.recalls(fcm1).mean())\n            fold_reports.append(dict(fold=int(fold), rows=int(hold.sum()), eligible_rows=int(score.sum()),\n                                     fit_rows=int(train.sum()), fit_class_counts=[int((y[train] == i).sum()) for i in gemini],\n                                     fit_prompts=len(fit_prompts), heldout_prompts=len(held_prompts), prompt_overlap=0,\n                                     specialist_iterations=iterations, baseline_ba=b0, candidate_ba=b1,\n                                     delta_pp=100 * (b1 - b0), **assignment.paired_counts(yt, p, q)))\n            support, difference = _conditional_draws(prompts[hold], yt, p, q, rng)\n            support_draws += support\n            difference_draws += difference\n\n    require(np.array_equal(rows_scored, eligible) and np.array_equal(candidate[~eligible], baseline[~eligible])\n            and np.array_equal(np.isin(candidate, gemini), np.isin(baseline, gemini))\n            and residual._digest(evidence) == score_sha and residual._digest(features) == feature_sha,\n            "stage13i_frozen_scores_or_routing_changed")\n    cm1 = reference.confusion(y, candidate)\n    support = cm0.sum(axis=1)\n    base_recall, cand_recall = reference.recalls(cm0), reference.recalls(cm1)\n    deltas = [Fraction(int(cm1[i, i]) - int(cm0[i, i]), int(support[i])) * 100 for i in range(K)]\n    delta = sum(deltas) / K\n    counts = assignment.paired_counts(y, baseline, candidate)\n    base_family, cand_family = _family_matrix(y, baseline, gemini), _family_matrix(y, candidate, gemini)\n    ecm0, ecm1 = reference.confusion(y[eligible], baseline[eligible]), reference.confusion(y[eligible], candidate[eligible])\n    n_eligible = int(eligible.sum())\n    require((support_draws > 0).all() and np.array_equal(support, cm1.sum(axis=1))\n            and np.array_equal(base_family, cand_family) and np.array_equal(cm1 - cm0, ecm1 - ecm0)\n            and int(ecm0.sum()) == int(ecm1.sum()) == n_eligible == sum(f["eligible_rows"] for f in fold_reports)\n            and counts["changed"] <= n_eligible and all(counts[k] == sum(f[k] for f in fold_reports) for k in counts),\n            "stage13i_paired_reconciliation")\n    interval = np.percentile((difference_draws / support_draws).mean(axis=1) * 100, [2.5, 97.5]).tolist()\n    criteria = dict(pooled_gain_at_least_half_pp=delta >= Fraction(1, 2), conditional_ci_lower_positive=interval[0] > 0,\n                    every_class_recall_loss_le_five_pp=min(deltas) >= -5)\n    true_eligible = int((eligible & true_gemini).sum())\n    correct_base, correct_candidate = int(ecm0.trace()), int(ecm1.trace())\n    return dict(baseline_ba=float(base_recall.mean()), candidate_ba=float(cand_recall.mean()), delta_pp=float(delta),\n                delta_pp_exact=str(delta), conditional_delta_ci95_pp=interval, class_supports=support.tolist(),\n                class_delta_pp=[float(d) for d in deltas], baseline_class_recall=base_recall.tolist(),\n                candidate_class_recall=cand_recall.tolist(), baseline_confusion_counts=cm0.tolist(),\n                candidate_confusion_counts=cm1.tolist(), baseline_confusion_sha256=digest,\n                candidate_confusion_sha256=reference.digest_json(cm1.tolist()),\n                eligible_rows=n_eligible, eligible_true_gemini_rows=true_eligible,\n                eligible_true_non_gemini_rows=n_eligible - true_eligible, eligible_baseline_errors=n_eligible - correct_base,\n                eligible_within_family_baseline_errors=true_eligible - correct_base,\n                eligible_baseline_correct=correct_base, eligible_candidate_correct=correct_candidate,\n                eligible_baseline_confusion_counts=ecm0.tolist(), eligible_candidate_confusion_counts=ecm1.tolist(),\n                gemini_classes=list(GEMINI_CLASSES), family_confusion_axis=["not_gemini", "gemini"],\n                baseline_family_confusion_counts=base_family.tolist(), candidate_family_confusion_counts=cand_family.tolist(),\n                folds=fold_reports, gate_uses_only_inferred_assignment_and_baseline_prediction=True,\n                outside_eligible_unchanged=True, predicted_family_unchanged=True,\n                fixed_features_and_log_evidence_unchanged=True, row_level_outputs=False,\n                specialist=dict(scaler="StandardScaler", estimator="multinomial logistic regression", parameters=deberta.HEAD.copy(),\n                                layers=[24, 6], fit_scope="true Gemini training rows only, outside the scoring prompt fold",\n                                application="replace variant by four-class argmax on gate rows only"),\n                bootstrap=dict(draws=reference.DRAWS, seed=reference.SEED,\n                               scope="conditional paired true-prompt resampling on fixed out-of-fold predictions"),\n                criteria=criteria, passes_development_screen=all(criteria.values()), **counts)\n', 'c3d51ccd238e9d111f66d7500be777b7a7095beeff4517547a1bfffd0c70b09e'), 'stage13h_residual_audit.py': ('"""Replay frozen Stage 13F and diagnose its residual errors; no variant."""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport time\n\nimport numpy as np\nfrom scipy import __version__ as scipy_version\nfrom sklearn import __version__ as sklearn_version\nfrom threadpoolctl import threadpool_limits\n\nimport stage13b_balancing_stress as reference\nimport stage13c_deberta_complement as stage13c\nimport stage13f_assignment_audit as stage13f\nimport stage13f_assignment_core as assignment\nimport stage13h_residual_core as residual\n\n\nCOMPLETE = "COMPLETE_STAGE13H_RESIDUAL_DIAGNOSTIC"\nFAILED = "FAILED_STAGE13H_RESIDUAL_DIAGNOSTIC"\nSCOPE = "stage13f_residual_errors_on_reused_prompt_heldout_glm_deberta_proxy"\nREPORT_NAME = "residual_audit_report.json"\nSTAGE13F_ANCHOR_VERSION = 368776259\nSTAGE13F_CONTROL_CM_SHA = "6b3a2cd77f8b7d5c24f343dd685e3be63aaa63b3635d59b6ccefc2528d1ae696"\nSTAGE13F_CONTROL_BA = 0.6146184439473029\nSTAGE13F_GEOMETRY_SHA = "6df9d668c3ccfb3140939a752ae1612f7d7fce775291877285581c0c9370c3f1"\nCONTROL_SIZES = list(range(12, 19))\nCONTROL_ROWS, CONTROL_CLUSTERS, CONTROL_CORRECT = 1889, 112, 1659\nRESIDUAL_ROWS = 810\nrequire = reference.require\n\n\ndef baseline_geometry_digest(geometry):\n    """Use the unchanged Stage 13G projection of the Stage 13F geometry."""\n    folds = [[fold["fold"], fold["rows"], fold["true_prompts"], fold["singleton_fallback"],\n              fold["eligible_rows"], fold["eligible_clusters"],\n              sorted((int(size), count) for size, count in fold["cluster_size_histogram"].items())]\n             for fold in geometry["folds"]]\n    return reference.digest_json(dict(folds=folds, rows=geometry["eligible_rows"],\n                                      clusters=geometry["eligible_clusters"]))\n\n\ndef verify_geometry(geometry, cluster_ids, folds):\n    """Lock the fold-local grouping before loading either score source."""\n    digest = baseline_geometry_digest(geometry)\n    require(digest == STAGE13F_GEOMETRY_SHA, "stage13h_stage13f_geometry_replay")\n    require(geometry["eligible_rows"] == CONTROL_ROWS and geometry["eligible_clusters"] == CONTROL_CLUSTERS\n            and geometry["eligible_mixed_prompt_clusters"] == 0\n            and geometry["eligible_clusters_with_duplicate_true_label"] == 0,\n            "stage13h_stage13f_baseline_coverage")\n    require(len(geometry["folds"]) == reference.FOLDS and cluster_ids.shape == folds.shape\n            and cluster_ids.dtype.kind in "iu" and (cluster_ids >= 0).all(), "stage13h_geometry_axes")\n    require(sum(int(size) * count for size, count in geometry["cluster_size_histogram"].items()\n                if int(size) < 12) == reference.ROWS - CONTROL_ROWS == RESIDUAL_ROWS,\n            "stage13h_residual_coverage")\n    return digest\n\n\ndef verify_diagnostic(diagnostic):\n    """Independently reconcile the frozen baseline and its diagnostic strata."""\n    assigned, unassigned = diagnostic["assigned"], diagnostic["unassigned"]\n    require(assigned["rows"] == CONTROL_ROWS and unassigned["rows"] == RESIDUAL_ROWS,\n            "stage13h_diagnostic_coverage")\n    matrix = np.asarray(diagnostic["baseline_confusion_counts"])\n    a = np.asarray(assigned["confusion_counts"])\n    u = np.asarray(unassigned["confusion_counts"])\n    require(all(m.shape == (reference.K, reference.K) and m.dtype.kind in "iu" and (m >= 0).all()\n                for m in (matrix, a, u)), "stage13h_diagnostic_matrices")\n    require(int(a.sum()) == CONTROL_ROWS and int(u.sum()) == RESIDUAL_ROWS\n            and np.array_equal(a + u, matrix), "stage13h_diagnostic_confusion_reconciliation")\n    require(int(np.trace(matrix)) == diagnostic["baseline_correct"] == CONTROL_CORRECT\n            and abs(float(reference.recalls(matrix).mean()) - STAGE13F_CONTROL_BA) < 1e-15\n            and abs(diagnostic["baseline_balanced_accuracy"] - STAGE13F_CONTROL_BA) < 1e-15\n            and reference.digest_json(matrix.tolist()) == diagnostic["baseline_confusion_sha256"]\n            == STAGE13F_CONTROL_CM_SHA, "stage13h_stage13f_control_confusion_replay")\n    require(all(s["correct"] == int(np.trace(m)) and s["errors"] == s["rows"] - s["correct"]\n                for s, m in ((assigned, a), (unassigned, u))), "stage13h_diagnostic_count_reconciliation")\n\n\ndef run(input_root, output_root):\n    started = time.monotonic()\n    root, out = Path(input_root), Path(output_root)\n    report = dict(schema_version=1, status=FAILED, scope=SCOPE, fit_attempted=False,\n                  fit_performed=False, cached_probabilities_loaded=False,\n                  stage13c_confusion_replayed=False, stage13f_geometry_replayed=False,\n                  stage13f_control_confusion_replayed=False, residual_diagnostic_performed=False,\n                  new_variant_scored=False, full_s07_oof_admitted=False, official_score_estimated=False,\n                  official_submission_created=False, competition_test_read=False,\n                  raw_rows_or_predictions_exported=False, uses_reused_development_prompts=True,\n                  s07_typography_guard_used=False, control_assignment_sizes=CONTROL_SIZES,\n                  stage13f_anchor_version=STAGE13F_ANCHOR_VERSION,\n                  expected_stage13f_control_confusion_sha256=STAGE13F_CONTROL_CM_SHA,\n                  expected_stage13f_geometry_sha256=STAGE13F_GEOMETRY_SHA,\n                  train_sha256=stage13f.TRAIN_SHA, train_ordered_text_sha256=stage13f.TEXT_SHA,\n                  canonical_prompt_fold_mapping_sha256=reference.FOLD_SHA,\n                  expected_input_version_ids=[stage13f.TRAIN_VERSION, reference.SOURCE_VERSION,\n                                              stage13c.DEBERTA_VERSION],\n                  expected_heldout_cache_sha256=reference.CACHE_SHA,\n                  expected_deberta_cache_sha256=stage13c.DEBERTA_SHA,\n                  sklearn_version=sklearn_version, numpy_version=np.__version__, scipy_version=scipy_version)\n    writable = False\n    try:\n        require(not out.exists() or (out.is_dir() and not any(out.iterdir())), "fresh_output_directory")\n        out.mkdir(parents=True, exist_ok=True)\n        writable = True\n        require(sklearn_version == stage13c.SKLEARN_VERSION, "stage13h_sklearn_version")\n        ids, prompts, texts, y, folds, classes, masks = stage13f.train_inputs(reference.one(root, "train.csv"))\n        with threadpool_limits(limits=stage13c.THREADS):\n            geometry, cluster_ids = assignment.geometry(texts, folds, prompts, y)\n        geometry_sha = verify_geometry(geometry, cluster_ids, folds)\n        report.update(geometry=geometry, rows=reference.ROWS, prompts=reference.PROMPTS,\n                      classes=classes.tolist(), stage13f_geometry_replayed=True,\n                      stage13f_geometry_sha256=geometry_sha, previously_untouched_rows=RESIDUAL_ROWS)\n\n        saved_ids, saved_prompts, saved_y, saved_folds, saved_classes, glm, producer = reference.cache_inputs(root)\n        require(all(np.array_equal(a, b) for a, b in zip((ids, prompts, y, folds, classes),\n                                                        (saved_ids, saved_prompts, saved_y, saved_folds, saved_classes))),\n                "stage13h_heldout_cache_row_or_fold_alignment")\n        report["cached_probabilities_loaded"] = True\n        stage13c.replay_control(glm, y, masks, producer)\n        report["glm_controls_replayed"] = True\n        features, id_mode = stage13c.load_features(reference.one(root, "enc_debv3l2_train.npz"), ids)\n        report["fit_attempted"] = True\n        deb, fit_details, pools = stage13c.fit_oof(features, y, masks)\n        report.update(fit_performed=True, deberta_fit_folds=fit_details, thread_pools=pools,\n                      deberta_id_verification=id_mode, feature_layers=[24, 6])\n        evidence, anchor_digest, residuals = stage13f.replay_stage13c(glm, deb, y, masks)\n        report.update(stage13c_confusion_replayed=True, stage13c_confusion_sha256=anchor_digest,\n                      mixture_weights={"glm": 0.75, "deberta": 0.25},\n                      balancing={"alpha": 1, "iterations": 100, "fold_local": True,\n                                 "maximum_column_mass_errors": residuals})\n        evidence_sha = hashlib.sha256(evidence.astype("<f8", copy=False).tobytes()).hexdigest()\n        diagnostic = residual.diagnose(evidence, cluster_ids, folds, prompts, y,\n                                       expected_baseline_confusion_sha256=STAGE13F_CONTROL_CM_SHA)\n        verify_diagnostic(diagnostic)\n        require(hashlib.sha256(evidence.astype("<f8", copy=False).tobytes()).hexdigest() == evidence_sha,\n                "stage13h_evidence_mutated")\n        require(hashlib.sha256(glm.astype("<f8", copy=False).tobytes()).hexdigest() == reference.PROB_SHA,\n                "stage13h_glm_cache_mutated")\n        report.update(status=COMPLETE, residual_diagnostic_performed=True, diagnostic=diagnostic,\n                      stage13f_control_confusion_replayed=True,\n                      stage13f_control_confusion_sha256=diagnostic["baseline_confusion_sha256"])\n    except reference.GateError as error:\n        report.update(failure_code=str(error))\n    except Exception as error:\n        report.update(failure_code="runtime_failure", failure_type=type(error).__name__)\n    report["seconds"] = round(time.monotonic() - started, 6)\n    if writable:\n        try:\n            saved = out / REPORT_NAME\n            tmp = out / ("." + REPORT_NAME + ".tmp")\n            tmp.write_text(json.dumps(report, sort_keys=True, indent=2, allow_nan=False) + "\\n", encoding="utf-8")\n            tmp.replace(saved)\n            require(json.loads(saved.read_text(encoding="utf-8")) == report, "stage13h_saved_report_mismatch")\n        except Exception as error:\n            report.update(status=FAILED, failure_code="report_write_or_readback_failure", failure_type=type(error).__name__)\n    return report\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--input-root", default="/kaggle/input")\n    parser.add_argument("--output-root", default="/kaggle/working/stage13h_residual_audit")\n    args = parser.parse_args()\n    result = run(args.input_root, args.output_root)\n    print("STAGE13H_RESIDUAL_AUDIT_JSON " + json.dumps(result, sort_keys=True, allow_nan=False), flush=True)\n    return 0 if result["status"] == COMPLETE else 2\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', '577a991478da210c5100fbad892168e5580c203a1fd9edb617dce0296d199c9e'), 'stage13h_residual_core.py': ('"""Describe the frozen Stage 13F decisions without changing any prediction."""\nfrom __future__ import annotations\n\nimport hashlib\n\nimport numpy as np\n\nimport stage13b_balancing_stress as reference\nimport stage13f_assignment_core as assignment\n\n\nK = reference.K\nMARGIN_CUTS = (0.10, 0.25, 0.50, 1.0, 2.0)\nPAIR_LIMIT = 15\nrequire = reference.require\n\n\ndef _digest(evidence):\n    return hashlib.sha256(evidence.astype("<f8", copy=False).tobytes()).hexdigest()\n\n\ndef _replay(evidence, cluster_ids, folds, prompts, targets, expected_digest):\n    assignment.validate_folds(folds)\n    assignment.validate_truth(folds, prompts, targets)\n    require(cluster_ids.shape == folds.shape and cluster_ids.dtype.kind in "iu"\n            and (cluster_ids >= 0).all(), "stage13h_cluster_axis")\n    require(evidence.shape == (len(folds), K) and evidence.dtype == np.float64\n            and np.isfinite(evidence).all(), "stage13h_log_evidence_axis")\n    require(np.max(np.abs(np.exp(evidence).sum(axis=1) - (1 + K * 1e-12))) <= 2e-12,\n            "stage13h_log_evidence_normalization")\n    inferred = assignment.buckets(cluster_ids)\n    require(max(map(len, inferred)) <= assignment.CAP\n            and all(np.unique(folds[idx]).size == 1 for idx in inferred),\n            "stage13h_cluster_boundaries")\n    dense = assignment.eligible(inferred)\n    require(dense, "stage13h_no_assignment_eligible_groups")\n    off = evidence.argmax(axis=1)\n    on = off.copy()\n    covered = np.zeros(len(folds), dtype=bool)\n    sizes = np.zeros(len(folds), dtype=np.int64)\n    for idx in inferred:\n        sizes[idx] = len(idx)\n    # Same solver, objective, group membership and original row order as Stage 13F.\n    for idx in dense:\n        r, c = assignment.linear_sum_assignment(-evidence[idx])\n        require(np.array_equal(r, np.arange(len(idx))) and len(np.unique(c)) == len(idx)\n                and not covered[idx].any(), "stage13h_assignment_unique_coverage")\n        on[idx[r]], covered[idx] = c, True\n    require(np.array_equal(off[~covered], on[~covered]) and (sizes > 0).all()\n            and np.array_equal(covered, sizes >= assignment.MIN_GROUP), "stage13h_assignment_partition")\n    matrix = reference.confusion(targets.astype(np.int64, copy=False), on)\n    digest = reference.digest_json(matrix.tolist())\n    require(digest == expected_digest, "stage13h_stage13f_on_confusion_replay")\n    return off, on, covered, sizes, inferred, matrix, digest\n\n\ndef _scores(matrix):\n    support = matrix.sum(axis=1)\n    correct = np.diag(matrix)\n    represented = support > 0\n    recalls = [int(c) / int(n) if n else None for c, n in zip(correct, support, strict=True)]\n    rows, n_correct = int(support.sum()), int(correct.sum())\n    return dict(rows=rows, correct=n_correct, errors=rows - n_correct,\n                row_accuracy=n_correct / rows if rows else None,\n                balanced_accuracy=float(np.mean(correct[represented] / support[represented]))\n                if represented.any() else None,\n                represented_classes=int(represented.sum()), class_supports=support.tolist(),\n                class_correct=correct.tolist(), class_recall=recalls)\n\n\ndef _pair_counts(matrix, y, decision, folds):\n    indices = [(i, j) for i in range(K) for j in range(K) if i != j and matrix[i, j]]\n    indices.sort(key=lambda ij: (-int(matrix[ij]), *ij))\n    out = []\n    for i, j in indices[:PAIR_LIMIT]:\n        selected = (y == i) & (decision == j)\n        n, supported = int(matrix[i, j]), int(matrix[i].sum())\n        out.append(dict(true_class_index=i, predicted_class_index=j, errors=n,\n                        true_class_support=supported, true_class_error_rate=n / supported,\n                        folds_with_error=int(np.unique(folds[selected]).size)))\n    return out\n\n\ndef _stratum(y, off, on, ranks, margin_bins, folds, *, detail):\n    cm = reference.confusion(y, on)\n    post = _scores(cm)\n    pre = _scores(reference.confusion(y, off))\n    misses = on != y\n    report = dict(rows=post["rows"], correct=post["correct"], errors=post["errors"],\n                  row_accuracy=post["row_accuracy"], balanced_accuracy=post["balanced_accuracy"],\n                  represented_classes=post["represented_classes"],\n                  pre_assignment_correct=pre["correct"], pre_assignment_errors=pre["errors"],\n                  pre_assignment_row_accuracy=pre["row_accuracy"],\n                  pre_assignment_balanced_accuracy=pre["balanced_accuracy"],\n                  evidence_true_top2_rows=int((ranks <= 2).sum()),\n                  evidence_true_top3_rows=int((ranks <= 3).sum()),\n                  errors_with_true_top2=int((misses & (ranks <= 2)).sum()),\n                  errors_with_true_top3=int((misses & (ranks <= 3)).sum()))\n    if detail:\n        margin = []\n        edges = (0.0, *MARGIN_CUTS)\n        for idx, lower in enumerate(edges):\n            in_bin = margin_bins == idx\n            n, error = int(in_bin.sum()), int((in_bin & misses).sum())\n            margin.append(dict(lower_inclusive=lower, upper_exclusive=edges[idx + 1] if idx < len(edges) - 1 else None,\n                               rows=n, correct=n - error, errors=error,\n                               errors_with_true_top2=int((in_bin & misses & (ranks <= 2)).sum()),\n                               errors_with_true_top3=int((in_bin & misses & (ranks <= 3)).sum())))\n        report.update(class_supports=post["class_supports"], class_correct=post["class_correct"],\n                      class_recall=post["class_recall"], confusion_counts=cm.tolist(),\n                      top_directed_error_pairs=_pair_counts(cm, y, on, folds),\n                      evidence_rank_counts=np.bincount(ranks, minlength=K + 1)[1:].tolist(),\n                      error_evidence_rank_counts=np.bincount(ranks[misses], minlength=K + 1)[1:].tolist(),\n                      evidence_margin_bins=margin)\n        require(sum(b["rows"] for b in margin) == post["rows"]\n                and sum(b["errors"] for b in margin) == post["errors"]\n                and sum(report["evidence_rank_counts"]) == post["rows"]\n                and sum(report["error_evidence_rank_counts"]) == post["errors"],\n                "stage13h_margin_or_rank_reconciliation")\n    return report\n\n\ndef diagnose(evidence, cluster_ids, folds, prompts, y, *, expected_baseline_confusion_sha256):\n    """Return only aggregate diagnostics after exact Stage 13F-on confusion replay."""\n    snapshot = _digest(evidence)\n    off, on, covered, sizes, inferred, baseline, digest = _replay(\n        evidence, cluster_ids, folds, prompts, y, expected_baseline_confusion_sha256)\n    y = y.astype(np.int64, copy=False)\n    rank_order = np.argsort(-evidence, axis=1, kind="stable")\n    require(np.array_equal(rank_order[:, 0], off), "stage13h_rank_tie_order")\n    ranks = (rank_order == y[:, None]).argmax(axis=1) + 1\n    top = evidence[np.arange(len(folds))[:, None], rank_order[:, :2]]\n    gap = top[:, 0] - top[:, 1]\n    require((gap >= 0).all() and np.isfinite(gap).all(), "stage13h_margin_domain")\n    margin_bins = np.searchsorted(MARGIN_CUTS, gap, side="right")\n\n    def summary(mask, *, detail=False):\n        return _stratum(y[mask], off[mask], on[mask], ranks[mask], margin_bins[mask], folds[mask], detail=detail)\n\n    assigned, unassigned = summary(covered, detail=True), summary(~covered, detail=True)\n    by_fold = [dict(fold=int(fold), assigned=summary((folds == fold) & covered),\n                    unassigned=summary((folds == fold) & ~covered)) for fold in np.unique(folds)]\n    by_size = []\n    for size in np.unique(sizes):\n        mask = sizes == size\n        part = summary(mask)\n        n_groups = sum(len(idx) == int(size) for idx in inferred)\n        require(n_groups * int(size) == part["rows"], "stage13h_size_reconciliation")\n        by_size.append(dict(cluster_size=int(size), clusters=n_groups,\n                            assignment_applied=bool(size >= assignment.MIN_GROUP), **part))\n    for key in ("rows", "correct", "errors", "pre_assignment_correct", "errors_with_true_top2", "errors_with_true_top3"):\n        require(all(sum(row[name][key] for row in by_fold) == part[key]\n                    and sum(row[key] for row in by_size if row["assignment_applied"] == (name == "assigned")) == part[key]\n                    for name, part in (("assigned", assigned), ("unassigned", unassigned))),\n                "stage13h_partition_reconciliation")\n    require(np.array_equal(np.asarray(assigned["confusion_counts"]) + np.asarray(unassigned["confusion_counts"]), baseline)\n            and unassigned["correct"] == unassigned["pre_assignment_correct"]\n            and unassigned["error_evidence_rank_counts"][0] == 0\n            and _digest(evidence) == snapshot, "stage13h_baseline_reconciliation")\n    base = _scores(baseline)\n    return dict(baseline_confusion_sha256=digest, baseline_confusion_counts=baseline.tolist(),\n                baseline_balanced_accuracy=base["balanced_accuracy"], baseline_correct=base["correct"],\n                assigned=assigned, unassigned=unassigned, by_fold=by_fold, by_cluster_size=by_size,\n                evidence_margin_definition="Highest minus second-highest frozen balanced natural log score, before assignment. Null upper bound means infinity.",\n                rank_definition="True model rank in frozen balanced scores before assignment, with ties by original class order. Rank count vectors are 1-based.",\n                balanced_accuracy_definition="Mean recall over classes represented in the stratum; absent class recall is null.",\n                fixed_log_evidence_unchanged=True, predictions_outside_assignment_unchanged=True,\n                row_level_outputs=False)\n', 'b5e1d9d45ca08b6970b82d296fd701e38ef8bd873f3891953f9b7a10a5e453d0'), 'stage13b_balancing_stress.py': ('"""Run one predeclared balancing diagnostic on the private GLM OOF cache."""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport time\n\nimport numpy as np\n\n\nCOMPLETE = "COMPLETE_GLM_SINKHORN_STRESS_ONLY"\nSCOPE = "glm_only_paired_fold_local_balancing_stress"\nSOURCE_VERSION = 368292220\nCACHE_SHA = "565df6f8e652a9850ff11cf4e8b6a81fcccbc7db593186ff7f53358e7190bd16"\nPROB_SHA = "41d80bc9350ab44d5aabe02b24e0174881a307bb1eb962f04196682e7faa94fe"\nID_SHA = "707edf645b0f09bba8f0983b5bc9f522595c95e8ca5b763b38babeb38200368d"\nLABEL_SHA = "088b14188775e41a314f269da3af74f96144a56caa242ccbeacc9e3c817a7e08"\nFOLD_SHA = "a55b035686cb7b31f6fbe852ac990668709aec4b913c5d0928d89d843bdbfe62"\nEXPECTED_RECALL = 61547 / 134100\nK, ROWS, PROMPTS, FOLDS = 18, 2699, 150, 5\nSCENARIOS = ("natural_full", "matched_balanced", "skew_a", "skew_b")\nDRAWS, SEED = 5000, 20261111\nATOL = 1e-12\n\n\nclass GateError(Exception):\n    pass\n\n\ndef require(condition, code):\n    if not bool(condition):\n        raise GateError(code)\n\n\ndef digest_json(obj):\n    return hashlib.sha256(json.dumps(obj, sort_keys=True, separators=(",", ":")).encode()).hexdigest()\n\n\ndef salted_hash(salt, value):\n    return hashlib.sha256((salt + "\\0" + str(value)).encode("utf-8")).digest()\n\n\ndef one(root, name):\n    matches = [p for p in root.rglob(name) if p.is_file()]\n    require(len(matches) == 1, "unique_" + name.replace(".", "_"))\n    return matches[0]\n\n\ndef validate_p(p):\n    require(p.ndim == 2 and p.shape[1] == K and p.dtype == np.float64 and len(p) > 0,\n            "probability_shape_or_dtype")\n    require(np.isfinite(p).all() and (p > 0).all() and (p <= 1).all(), "invalid_probability_values")\n    require(np.max(np.abs(p.sum(axis=1) - 1.0)) <= ATOL, "probability_row_sums")\n\n\ndef evidence(p, uniform):\n    """Use the source\'s fixed 100 updates and branch threshold on existing P."""\n    validate_p(p)\n    if not uniform or len(p) < K:\n        return np.log(p + 1e-12), None\n    q = p.copy()\n    with np.errstate(divide="raise", invalid="raise", over="raise"):\n        for _ in range(100):\n            q *= (len(q) / 18) / q.sum(0, keepdims=True)\n            q /= q.sum(1, keepdims=True)\n    validate_p(q)\n    residual = float(np.max(np.abs(q.sum(axis=0) - len(q) / K)))\n    require(np.isfinite(residual), "nonfinite_column_mass_error")\n    return np.log(q + 1e-12), residual\n\n\ndef cache_inputs(root):\n    cache = one(root, "heldout_probabilities.npz")\n    report = one(root, "readout_report.json")\n    require(cache.parent == report.parent, "cache_report_parent_mismatch")\n    with cache.open("rb") as f:\n        require(hashlib.file_digest(f, "sha256").hexdigest() == CACHE_SHA, "cache_byte_hash_mismatch")\n    r = json.loads(report.read_text(encoding="utf-8"))\n    require(r.get("status") == "COMPLETE_GLM_PLAIN_GROUPED_DEVELOPMENT_REFERENCE"\n            and r.get("scope") == "glm_only_prompt_heldout_development"\n            and r.get("heldout_cache_sha256") == CACHE_SHA\n            and r.get("probability_values_sha256") == PROB_SHA\n            and r.get("admission_checks_passed") == 7, "producer_report_contract")\n    for flag in ("fit_performed", "heldout_predictions_produced", "heldout_cache_readback_passed"):\n        require(r.get(flag) is True, "producer_report_missing_required_flag")\n    for flag in ("balancing_performed", "official_score_estimated", "full_s07_oof_admitted", "raw_rows_or_predictions_exported"):\n        require(r.get(flag) is False, "producer_report_scope")\n    keys = ("ids", "prompt_ids", "targets", "folds", "classes", "probabilities")\n    with np.load(cache, allow_pickle=False) as z:\n        require(len(z.files) == len(keys) and set(z.files) == set(keys), "cache_schema")\n        ids, prompts, targets, folds, classes, p = (z[k] for k in keys)\n    require(ids.shape == prompts.shape == targets.shape == folds.shape == (ROWS,), "cache_row_axes")\n    require(ids.dtype.kind == prompts.dtype.kind == classes.dtype.kind == "U"\n            and targets.dtype == folds.dtype == np.int64 and classes.shape == (K,), "cache_axis_dtypes")\n    require(len(np.unique(ids)) == ROWS and len(np.unique(prompts)) == PROMPTS, "cache_unique_axes")\n    require(digest_json(ids.tolist()) == ID_SHA and digest_json(classes.tolist()) == LABEL_SHA, "cache_axis_hashes")\n    require(np.array_equal(np.unique(targets), np.arange(K)) and np.array_equal(np.unique(folds), np.arange(1, 6)),\n            "cache_class_or_fold_indices")\n    require(np.bincount(targets).tolist() == [149] + [150] * 17, "original_class_supports")\n    validate_p(p)\n    require(len(p) == ROWS and hashlib.sha256(p.astype("<f8", copy=False).tobytes()).hexdigest() == PROB_SHA,\n            "cache_probability_hash")\n    mapping = {}\n    for group, fold in zip(prompts, folds):\n        require(mapping.setdefault(str(group), int(fold)) == int(fold), "prompt_crosses_fold")\n    require(digest_json(mapping) == FOLD_SHA and r.get("canonical_prompt_fold_mapping_sha256") == FOLD_SHA,\n            "canonical_fold_map_hash")\n    require(r.get("classes") == classes.tolist() and len(r.get("folds", [])) == FOLDS, "producer_fold_axis")\n    for f, item in enumerate(r["folds"], 1):\n        n = 540 if f < 5 else 539\n        mask = folds == f\n        require(item.get("fold") == f and int(mask.sum()) == n and len(np.unique(prompts[mask])) == 30\n                and [item.get(k) for k in ("fit_rows", "valid_rows", "fit_prompts", "valid_prompts", "prompt_overlap")]\n                == [ROWS - n, n, 120, 30, 0], "producer_fold_exclusion_contract")\n    return ids, prompts, targets, folds, classes, p, r\n\n\ndef mask_indices(ids, targets, classes, indices):\n    """Rank once per class. Use matched subsets for all paired scoring arms."""\n    ranked = sorted(range(K), key=lambda c: (salted_hash("stage13b-class-v1", classes[c]), str(classes[c]).encode("utf-8")))\n    a_major = set(ranked[:9])\n    buckets = [sorted(indices[targets[indices] == c].tolist(),\n                      key=lambda i: (salted_hash("stage13b-row-v1", ids[i]), str(ids[i]).encode("utf-8"))) for c in range(K)]\n    require(all(len(rows) >= 27 for rows in buckets), "insufficient_mask_class_support")\n    masks = {"natural_full": indices}\n    for name in SCENARIOS[1:]:\n        counts = [20] * K if name == "matched_balanced" else [\n            27 if ((c in a_major) == (name == "skew_a")) else 13 for c in range(K)]\n        chosen = np.asarray(sorted(i for c, rows in enumerate(buckets) for i in rows[:counts[c]]), dtype=np.int64)\n        require(len(chosen) == len(np.unique(chosen)) == 360 and np.bincount(targets[chosen], minlength=K).tolist() == counts,\n                "matched_mask_supports")\n        masks[name] = chosen\n    return masks\n\n\ndef confusion(y, pred):\n    return np.bincount(y * K + pred, minlength=K*K).reshape(K, K)\n\n\ndef recalls(counts):\n    totals = counts.sum(axis=1)\n    require((totals > 0).all(), "class_missing_from_score")\n    return counts.diagonal() / totals\n\n\ndef bootstrap_delta(prompt_index, targets, correct_off, correct_uniform, weights):\n    supports = np.zeros((30, K), dtype=np.int64)\n    differences = np.zeros_like(supports)\n    np.add.at(supports, (prompt_index, targets), 1)\n    np.add.at(differences, (prompt_index, targets), correct_off.astype(np.int64) - correct_uniform.astype(np.int64))\n    return weights @ supports, weights @ differences\n\n\ndef diagnostics(ids, prompts, targets, folds, classes, p, producer):\n    # Freeze the shared paired resampling before applying either arm.\n    rng = np.random.default_rng(SEED)\n    weights = [rng.multinomial(30, np.full(30, 1/30), size=DRAWS) for _ in range(FOLDS)]\n    stores = {n: {"off": np.zeros((K, K), dtype=np.int64), "uniform": np.zeros((K, K), dtype=np.int64),\n                  "support_draws": np.zeros((DRAWS, K), dtype=np.int64), "delta_draws": np.zeros((DRAWS, K), dtype=np.int64),\n                  "fixed": 0, "broken": 0, "changed": 0, "batches": []} for n in SCENARIOS}\n    digest_before = hashlib.sha256(p.astype("<f8", copy=False).tobytes()).hexdigest()\n    for f in range(1, 6):\n        indices = np.flatnonzero(folds == f)\n        ordered_groups = np.unique(prompts[indices])\n        require(len(ordered_groups) == 30, "scoring_batch_prompt_count")\n        for name, selected in mask_indices(ids, targets, classes, indices).items():\n            require(len(np.unique(prompts[selected])) == 30 and (folds[selected] == f).all(), "masked_batch_fold_or_prompt_count")\n            y, pp = targets[selected], p[selected]\n            log_off, _ = evidence(pp, False)\n            log_uniform, residual = evidence(pp, True)\n            pred_off, pred_uniform = log_off.argmax(1), log_uniform.argmax(1)\n            off, uniform = confusion(y, pred_off), confusion(y, pred_uniform)\n            co, cu = pred_off == y, pred_uniform == y\n            fixed, broken = int((co & ~cu).sum()), int((cu & ~co).sum())\n            changed = int((pred_off != pred_uniform).sum())\n            require(fixed + broken <= changed and int(co.sum() - cu.sum()) == fixed - broken, "paired_counts")\n            b = stores[name]\n            b["off"] += off\n            b["uniform"] += uniform\n            b["fixed"] += fixed\n            b["broken"] += broken\n            b["changed"] += changed\n            b["batches"].append(dict(fold=f, rows=len(y), prompts=30, class_supports=off.sum(1).tolist(),\n                                     off_ba=float(recalls(off).mean()), uniform_ba=float(recalls(uniform).mean()),\n                                     uniform_max_column_mass_error=residual))\n            ds, dd = bootstrap_delta(np.searchsorted(ordered_groups, prompts[selected]), y, co, cu, weights[f-1])\n            b["support_draws"] += ds\n            b["delta_draws"] += dd\n    natural = stores["natural_full"]["off"]\n    require(np.array_equal(natural, np.asarray(producer["confusion_counts"]))\n            and abs(float(recalls(natural).mean()) - EXPECTED_RECALL) <= 1e-15\n            and abs(float(producer["pooled_balanced_accuracy"]) - EXPECTED_RECALL) <= 1e-15, "original_plain_reference_replay")\n    require(hashlib.sha256(p.astype("<f8", copy=False).tobytes()).hexdigest() == digest_before, "mutated_upstream_probabilities")\n    results = {}\n    for name, b in stores.items():\n        off, uniform = b["off"], b["uniform"]\n        ro, ru = recalls(off), recalls(uniform)\n        require(np.array_equal(off.sum(1), uniform.sum(1)) and (b["support_draws"] > 0).all(), "paired_support_or_bootstrap_coverage")\n        intervals = np.percentile((b["delta_draws"] / b["support_draws"]).mean(axis=1) * 100, [2.5, 97.5])\n        results[name] = dict(rows=int(off.sum()), batches=b["batches"], off_ba=float(ro.mean()), uniform_ba=float(ru.mean()),\n                             delta_pp=float((ro.mean() - ru.mean()) * 100), conditional_delta_ci95_pp=intervals.tolist(),\n                             class_supports=off.sum(1).tolist(), class_delta_pp=((ro - ru) * 100).tolist(),\n                             off_class_recalls=ro.tolist(), uniform_class_recalls=ru.tolist(),\n                             off_confusion_counts=off.tolist(), uniform_confusion_counts=uniform.tolist(),\n                             corrected_by_off=b["fixed"], broken_by_off=b["broken"], changed_by_off=b["changed"])\n    require([results[n]["rows"] for n in SCENARIOS] == [2699, 1800, 1800, 1800], "scenario_row_counts")\n    criteria = dict(\n        both_skew_point_gains_ge_half_pp=all(results[n]["delta_pp"] >= 0.5 for n in ("skew_a", "skew_b")),\n        both_skew_conditional_ci_lowers_positive=all(results[n]["conditional_delta_ci95_pp"][0] > 0 for n in ("skew_a", "skew_b")),\n        natural_and_matched_losses_le_half_pp=all(results[n]["delta_pp"] >= -0.5 for n in ("natural_full", "matched_balanced")),\n        every_class_recall_loss_le_five_pp=all(min(b["class_delta_pp"]) >= -5.0 - 1e-12 for b in results.values()),\n    )\n    return results, criteria\n\n\ndef run(input_root, output_root):\n    t = time.monotonic()\n    root, out = Path(input_root), Path(output_root)\n    report = dict(schema_version=1, status="FAILED_GLM_SINKHORN_STRESS", scope=SCOPE, source_version_id=SOURCE_VERSION,\n                  heldout_cache_sha256=CACHE_SHA, fit_performed=False, official_score_estimated=False,\n                  full_s07_oof_admitted=False, raw_rows_or_predictions_exported=False)\n    writable = False\n    try:\n        require(not out.exists() or (out.is_dir() and not any(out.iterdir())), "fresh_output_directory")\n        out.mkdir(parents=True, exist_ok=True)\n        writable = True\n        ids, groups, targets, folds, classes, p, prior = cache_inputs(root)\n        results, criteria = diagnostics(ids, groups, targets, folds, classes, p, prior)\n        report.update(status=COMPLETE, scenarios=results, classes=classes.tolist(), decision_criteria=criteria,\n                      passes_proxy_screen=all(criteria.values()), probabilities_unchanged=True,\n                      original_plain_reference_replayed=True, actual_folds=5, prompts=150,\n                      bootstrap=dict(draws=DRAWS, seed=SEED, scope="fixed-prediction stratified prompt resampling"),\n                      control=dict(alpha=1, iterations=100, minimum_batch_rows=18), candidate=dict(alpha=0),\n                      delta_direction="off_minus_uniform", numpy_version=np.__version__)\n    except GateError as error:\n        report.update(failure_code=str(error))\n    except Exception as error:\n        report.update(failure_code="runtime_failure", failure_type=type(error).__name__)\n    report["seconds"] = round(time.monotonic() - t, 6)\n    if writable:\n        try:\n            tmp = out / ".balancing_stress_report.tmp"\n            tmp.write_text(json.dumps(report, sort_keys=True, indent=2, allow_nan=False) + "\\n", encoding="utf-8")\n            tmp.replace(out / "balancing_stress_report.json")\n        except Exception as error:\n            report.update(status="FAILED_GLM_SINKHORN_STRESS", failure_code="report_write_failure", failure_type=type(error).__name__)\n    return report\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--input-root", default="/kaggle/input")\n    parser.add_argument("--output-root", default="/kaggle/working/stage13b_balancing_stress")\n    args = parser.parse_args()\n    result = run(args.input_root, args.output_root)\n    print("STAGE13B_GLM_STRESS_JSON_BEGIN", flush=True)\n    print(json.dumps(result, sort_keys=True, allow_nan=False), flush=True)\n    print("STAGE13B_GLM_STRESS_JSON_END", flush=True)\n    return 0 if result["status"] == COMPLETE else 2\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'f4981f385074f60ee034b979d4aa6672c7d28eaa408c374d5caf55a7c446fbd9'), 'stage13c_deberta_complement.py': ('"""Stage 13C: one fixed DeBERTa readout and paired, fold-local GLM comparison."""\nfrom __future__ import annotations\n\nimport argparse\nfrom fractions import Fraction\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nimport pickletools\nimport time\nimport warnings\nimport zipfile\n\nimport numpy as np\nfrom sklearn import __version__ as sklearn_version\nfrom sklearn.exceptions import ConvergenceWarning\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.preprocessing import StandardScaler\nfrom threadpoolctl import threadpool_info, threadpool_limits\n\nimport stage13b_balancing_stress as reference\n\n\nCOMPLETE = "COMPLETE_GLM_DEBERTA_COMPLEMENT_SCREEN"\nFAILED = "FAILED_GLM_DEBERTA_COMPLEMENT_SCREEN"\nSCOPE = "glm_deberta_paired_fold_local_development"\nDEBERTA_VERSION = 358745836\nDEBERTA_SHA = "11eed58762ad128f97457dde9cc4ab03c7932e1b4b04845ec6664c834b1b1dda"\nDEBERTA_BYTES = 99537639\nRAW_CONTROL_CM_SHA = "fa3ced3a896ce9389bf0427a42a30d8e7a30089c792cb92b8988953204b98747"\nUNIFORM_CONTROL_CM_SHA = "e81a80974d67dc60dc93faf3144dff56ab2055a327abcf7805dbb3556f364b0b"\nUNIFORM_CONTROL_BA = 0.4608252547849864\nSKLEARN_VERSION = "1.6.1"\nTHREADS = 2\nHEAD = dict(C=0.005, solver="lbfgs", max_iter=5000, tol=1e-4, fit_intercept=True, random_state=0)\nK, ROWS, FOLDS = reference.K, reference.ROWS, reference.FOLDS\nrequire = reference.require\n\n\ndef header(stream):\n    """Check the NPY metadata before array materialization."""\n    version = np.lib.format.read_magic(stream)\n    require(version in ((1, 0), (2, 0)), "unsupported_deberta_npy_version")\n    read = np.lib.format.read_array_header_1_0 if version == (1, 0) else np.lib.format.read_array_header_2_0\n    return read(stream, max_header_size=4096)\n\n\ndef checked_ids(blob, expected):\n    """Verify the frozen ID member without deserializing any object array."""\n    stream = io.BytesIO(blob)\n    shape, fortran, dtype = header(stream)\n    require(shape == expected.shape and not fortran, "deberta_ids_shape_or_order")\n    if not dtype.hasobject:\n        require(dtype.kind in "SU", "deberta_ids_numeric_or_invalid_dtype")\n        actual = np.load(io.BytesIO(blob), allow_pickle=False).astype(str)\n        mode = "numpy_nonobject"\n    else:\n        require(dtype == np.dtype(object), "deberta_ids_object_dtype")\n        payload = stream.read()\n        id_set = set(expected.tolist())\n        literals = []\n        # The byte hash is checked before opening the archive. genops parses\n        # bytecode, never imports symbols or executes a pickle instruction.\n        last = None\n        for n, (op, value, pos) in enumerate(pickletools.genops(payload), 1):\n            require(n <= 100000, "deberta_ids_opcode_limit")\n            last = (op.name, pos)\n            if isinstance(value, bytes):\n                try:\n                    value = value.decode("ascii")\n                except UnicodeDecodeError:\n                    continue\n            if isinstance(value, str) and value in id_set:\n                literals.append(value)\n        require(last == ("STOP", len(payload) - 1), "deberta_ids_pickle_trailing_data")\n        actual = np.asarray(literals, dtype=str)\n        mode = "object_npy_static_unique_ID_literals_no_unpickling"\n    require(np.array_equal(actual, expected), "deberta_ids_order_mismatch")\n    return mode\n\n\ndef load_features(path, ids):\n    with path.open("rb") as f:\n        require(path.stat().st_size == DEBERTA_BYTES, "deberta_cache_size_mismatch")\n        require(hashlib.file_digest(f, "sha256").hexdigest() == DEBERTA_SHA, "deberta_cache_hash_mismatch")\n        f.seek(0)\n        with zipfile.ZipFile(f) as z:\n            for key, limit in (("ids.npy", 2_000_000), ("layers.npy", 4096),\n                               ("emb.npy", ROWS * 9 * 1024 * 2 + 4096)):\n                require(z.namelist().count(key) == 1 and z.getinfo(key).file_size < limit,\n                        "deberta_cache_member_bounds")\n            mode = checked_ids(z.read("ids.npy"), ids)\n            with z.open("emb.npy") as member:\n                shape, _, dtype = header(member)\n                require(shape == (ROWS, 9, 1024) and dtype == np.dtype(np.float16), "deberta_emb_shape_or_dtype")\n            with z.open("layers.npy") as member:\n                shape, _, dtype = header(member)\n                require(shape == (9,) and dtype.kind in "iu", "deberta_layers_shape_or_dtype")\n        f.seek(0)\n        with np.load(f, allow_pickle=False) as z:\n            layers, emb = z["layers"], z["emb"]\n            require(layers.shape == (9,) and layers.dtype.kind in "iu" and len(np.unique(layers)) == 9,\n                    "deberta_layers_invalid")\n            require(all((layers == layer).sum() == 1 for layer in (24, 6)), "deberta_missing_fixed_layers")\n            require(emb.shape == (ROWS, 9, 1024) and emb.dtype == np.float16 and np.isfinite(emb).all(),\n                    "deberta_emb_validation")\n            x = np.ascontiguousarray(np.concatenate([emb[:, np.flatnonzero(layers == layer).item()]\n                                                     for layer in (24, 6)], axis=1), dtype=np.float64)\n    require(x.shape == (ROWS, 2048) and np.isfinite(x).all(), "deberta_selected_features")\n    return x, mode\n\n\ndef fold_masks(prompts, y, folds):\n    """Validate all fitting/scoring boundaries before fitting any head."""\n    require(prompts.shape == y.shape == folds.shape == (ROWS,), "complement_axis_shape")\n    masks = []\n    for f in range(1, FOLDS + 1):\n        val = folds == f\n        fit_prompts, val_prompts = np.unique(prompts[~val]), np.unique(prompts[val])\n        require(len(fit_prompts) == 120 and len(val_prompts) == 30\n                and len(np.intersect1d(fit_prompts, val_prompts)) == 0\n                and int(val.sum()) == (540 if f < 5 else 539), "complement_fold_exclusion")\n        require(all(np.array_equal(np.unique(y[m]), np.arange(K)) for m in (val, ~val)),\n                "complement_fold_class_coverage")\n        masks.append(val)\n    require(np.asarray(masks).sum(axis=0).tolist() == [1] * ROWS, "complement_fold_cover")\n    return masks\n\n\ndef fit_oof(x, y, masks):\n    require(x.ndim == 2 and len(x) == ROWS and x.dtype == np.float64 and np.isfinite(x).all(),\n            "complement_fit_features")\n    p = np.full((ROWS, K), np.nan, dtype=np.float64)\n    filled = np.zeros(ROWS, dtype=np.int8)\n    details = []\n    with threadpool_limits(limits=THREADS):\n        pools = [{"api": item["user_api"], "threads": item["num_threads"]} for item in threadpool_info()]\n        require(all(0 < pool["threads"] <= THREADS for pool in pools), "complement_thread_limit")\n        for fold, val in enumerate(masks, 1):\n            started = time.monotonic()\n            scaler = StandardScaler().fit(x[~val])\n            with warnings.catch_warnings():\n                warnings.simplefilter("error", ConvergenceWarning)\n                estimator = LogisticRegression(**HEAD).fit(scaler.transform(x[~val]), y[~val])\n            require(np.array_equal(estimator.classes_, np.arange(K)), "complement_estimator_class_axis")\n            niter = [int(n) for n in estimator.n_iter_]\n            require(len(niter) == 1 and 0 < niter[0] < HEAD["max_iter"], "complement_iteration_limit")\n            scores = estimator.predict_proba(scaler.transform(x[val]))\n            reference.validate_p(scores)\n            require(len(scores) == val.sum() and (filled[val] == 0).all(), "complement_duplicate_or_missing_rows")\n            p[val], filled[val] = scores, 1\n            details.append(dict(fold=fold, iterations=niter[0], fit_rows=int((~val).sum()),\n                                valid_rows=int(val.sum()), fit_prompts=120, valid_prompts=30,\n                                prompt_overlap=0, seconds=round(time.monotonic() - started, 6)))\n    require((filled == 1).all() and len(details) == FOLDS, "complement_oof_cover")\n    reference.validate_p(p)\n    return p, details, pools\n\n\ndef replay_control(p, y, masks, producer):\n    """Require the full prior confusion evidence before fitting anything."""\n    predicted = np.full(ROWS, -1, dtype=np.int64)\n    residuals = []\n    for mask in masks:\n        scores, residual = reference.evidence(p[mask], True)\n        predicted[mask] = scores.argmax(axis=1)\n        residuals.append(residual)\n    require((predicted >= 0).all(), "control_oof_cover")\n    raw = reference.confusion(y, reference.evidence(p, False)[0].argmax(axis=1))\n    uniform = reference.confusion(y, predicted)\n    require(raw.tolist() == producer["confusion_counts"]\n            and reference.digest_json(raw.tolist()) == RAW_CONTROL_CM_SHA\n            and abs(float(reference.recalls(raw).mean()) - reference.EXPECTED_RECALL) < 1e-15,\n            "complement_raw_control_confusion_mismatch")\n    require(reference.digest_json(uniform.tolist()) == UNIFORM_CONTROL_CM_SHA\n            and abs(float(reference.recalls(uniform).mean()) - UNIFORM_CONTROL_BA) < 1e-15,\n            "complement_uniform_control_confusion_mismatch")\n    return predicted, uniform, residuals\n\n\ndef paired_scores(glm, deb, prompts, y, masks, control, control_cm, residuals, folds):\n    """Apply the same frozen balancing to full, separate scoring folds."""\n    candidate = np.full(ROWS, -1, dtype=np.int64)\n    alone = np.full(ROWS, -1, dtype=np.int64)\n    rng = np.random.default_rng(reference.SEED)\n    bootstrap_support = np.zeros((reference.DRAWS, K), dtype=np.int64)\n    bootstrap_diff = np.zeros_like(bootstrap_support)\n    for f, mask in enumerate(masks):\n        pp = 0.75 * glm[mask] + 0.25 * deb[mask]\n        scores, candidate_residual = reference.evidence(pp, True)\n        candidate[mask] = scores.argmax(axis=1)\n        alone_scores, _ = reference.evidence(deb[mask], True)\n        alone[mask] = alone_scores.argmax(axis=1)\n        yt, gc, mix = y[mask], control[mask], candidate[mask]\n        cg, cc = reference.confusion(yt, gc), reference.confusion(yt, mix)\n        cgood, ggood = mix == yt, gc == yt\n        ordered = np.unique(prompts[mask])\n        weights = rng.multinomial(30, np.full(30, 1 / 30), size=reference.DRAWS)\n        support, diff = reference.bootstrap_delta(np.searchsorted(ordered, prompts[mask]), yt, cgood, ggood, weights)\n        bootstrap_support += support\n        bootstrap_diff += diff\n        folds[f].update(control_ba=float(reference.recalls(cg).mean()),\n                        candidate_ba=float(reference.recalls(cc).mean()),\n                        control_column_mass_error=residuals[f],\n                        candidate_column_mass_error=candidate_residual,\n                        corrected_by_candidate=int((cgood & ~ggood).sum()),\n                        broken_by_candidate=int((ggood & ~cgood).sum()))\n    require((bootstrap_support > 0).all(), "complement_bootstrap_missing_class")\n    require((candidate >= 0).all() and (alone >= 0).all(), "complement_predictions_incomplete")\n    cand_cm, alone_cm = reference.confusion(y, candidate), reference.confusion(y, alone)\n    raw_deb_cm = reference.confusion(y, reference.evidence(deb, False)[0].argmax(axis=1))\n    controls, candidates = reference.recalls(control_cm), reference.recalls(cand_cm)\n    supports = control_cm.sum(axis=1)\n    require(np.array_equal(supports, cand_cm.sum(axis=1)), "complement_support_mismatch")\n    exact = [Fraction(int(cand_cm[i, i]) - int(control_cm[i, i]), int(supports[i])) * 100 for i in range(K)]\n    delta = sum(exact) / K\n    ci = np.percentile((bootstrap_diff / bootstrap_support).mean(axis=1) * 100, [2.5, 97.5]).tolist()\n    corrected = int(((candidate == y) & (control != y)).sum())\n    broken = int(((control == y) & (candidate != y)).sum())\n    changed = int((candidate != control).sum())\n    require(corrected + broken <= changed\n            and corrected - broken == int(cand_cm.trace() - control_cm.trace())\n            and corrected == sum(f["corrected_by_candidate"] for f in folds)\n            and broken == sum(f["broken_by_candidate"] for f in folds), "complement_paired_count_mismatch")\n    criteria = dict(pooled_gain_at_least_half_pp=delta >= Fraction(1, 2),\n                    conditional_ci_lower_positive=ci[0] > 0,\n                    every_class_recall_loss_le_five_pp=min(exact) >= -5)\n    return dict(control_ba=float(controls.mean()), candidate_ba=float(candidates.mean()),\n                deberta_alone_ba=float(reference.recalls(alone_cm).mean()),\n                deberta_alone_raw_ba=float(reference.recalls(raw_deb_cm).mean()),\n                delta_pp=float(delta), delta_pp_exact=str(delta), conditional_delta_ci95_pp=ci,\n                class_supports=supports.tolist(), class_delta_pp=[float(v) for v in exact],\n                control_class_recalls=controls.tolist(), candidate_class_recalls=candidates.tolist(),\n                control_confusion_counts=control_cm.tolist(), candidate_confusion_counts=cand_cm.tolist(),\n                deberta_alone_confusion_counts=alone_cm.tolist(),\n                deberta_alone_raw_confusion_counts=raw_deb_cm.tolist(),\n                corrected_by_candidate=corrected, broken_by_candidate=broken, changed_by_candidate=changed,\n                decision_criteria=criteria, passes_proxy_screen=all(criteria.values()))\n\n\ndef run(input_root, output_root):\n    started = time.monotonic()\n    root, out = Path(input_root), Path(output_root)\n    report = dict(schema_version=1, status=FAILED, scope=SCOPE, fit_attempted=False, fit_performed=False,\n                  raw_rows_or_predictions_exported=False, official_score_estimated=False,\n                  full_s07_oof_admitted=False, expected_input_version_ids=[reference.SOURCE_VERSION, DEBERTA_VERSION],\n                  heldout_cache_sha256=reference.CACHE_SHA, deberta_cache_sha256=DEBERTA_SHA)\n    writable = False\n    try:\n        require(not out.exists() or (out.is_dir() and not any(out.iterdir())), "fresh_output_directory")\n        out.mkdir(parents=True, exist_ok=True)\n        writable = True\n        require(sklearn_version == SKLEARN_VERSION, "complement_sklearn_version")\n        ids, prompts, y, folds, classes, glm, producer = reference.cache_inputs(root)\n        masks = fold_masks(prompts, y, folds)\n        control, cm, residuals = replay_control(glm, y, masks, producer)\n        report["control_replayed"] = True\n        features, id_mode = load_features(reference.one(root, "enc_debv3l2_train.npz"), ids)\n        report["fit_attempted"] = True\n        deb, details, pools = fit_oof(features, y, masks)\n        report["fit_performed"] = True\n        scores = paired_scores(glm, deb, prompts, y, masks, control, cm, residuals, details)\n        require(hashlib.sha256(glm.astype("<f8", copy=False).tobytes()).hexdigest() == reference.PROB_SHA,\n                "complement_glm_mutated")\n        report.update(status=COMPLETE, rows=ROWS, prompts=reference.PROMPTS, classes=classes.tolist(), folds=details,\n                      estimator=HEAD, thread_pools=pools, feature_layers=[24, 6], feature_dtype="float16_cached",\n                      deberta_id_verification=id_mode, sklearn_version=sklearn_version, numpy_version=np.__version__,\n                      control_raw_confusion_sha256=RAW_CONTROL_CM_SHA, control_uniform_confusion_sha256=UNIFORM_CONTROL_CM_SHA,\n                      mixture_weights={"glm": 0.75, "deberta": 0.25},\n                      balancing={"alpha": 1, "iterations": 100, "minimum_batch_rows": 18, "fold_local": True},\n                      bootstrap={"draws": reference.DRAWS, "seed": reference.SEED,\n                                 "scope": "conditional stratified paired prompt resampling on fixed predictions"},\n                      **scores)\n    except reference.GateError as error:\n        report.update(failure_code=str(error))\n    except Exception as error:\n        report.update(failure_code="runtime_failure", failure_type=type(error).__name__)\n    report["seconds"] = round(time.monotonic() - started, 6)\n    if writable:\n        try:\n            tmp = out / ".complement_report.tmp"\n            tmp.write_text(json.dumps(report, sort_keys=True, indent=2, allow_nan=False) + "\\n", encoding="utf-8")\n            tmp.replace(out / "complement_report.json")\n        except Exception as error:\n            report.update(status=FAILED, failure_code="report_write_failure", failure_type=type(error).__name__)\n    return report\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--input-root", default="/kaggle/input")\n    parser.add_argument("--output-root", default="/kaggle/working/stage13c_deberta_complement")\n    args = parser.parse_args()\n    result = run(args.input_root, args.output_root)\n    print("STAGE13C_GLM_DEBERTA_JSON_BEGIN", flush=True)\n    print(json.dumps(result, sort_keys=True, allow_nan=False), flush=True)\n    print("STAGE13C_GLM_DEBERTA_JSON_END", flush=True)\n    return 0 if result["status"] == COMPLETE else 2\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'b9b6f523b8a6b1f9f1dae556c863f7fee34f56fddfb27adb0946021e1f17181e'), 'stage13f_assignment_audit.py': ('"""Stage 13F: test S07\'s dense-group assignment on a reduced held-out proxy."""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport time\n\nimport numpy as np\nimport pandas as pd\nfrom scipy import __version__ as scipy_version\nfrom sklearn import __version__ as sklearn_version\nfrom sklearn.model_selection import GroupKFold\nfrom threadpoolctl import threadpool_limits\n\nimport stage13b_balancing_stress as reference\nimport stage13c_deberta_complement as prior\nimport stage13f_assignment_core as assignment\n\n\nCOMPLETE_ZERO = "COMPLETE_ZERO_COVERAGE"\nCOMPLETE_PAIRED = "COMPLETE_PAIRED_ASSIGNMENT_PROXY"\nFAILED = "FAILED_STAGE13F_ASSIGNMENT_AUDIT"\nSCOPE = "s07_assignment_rule_on_reused_prompt_heldout_glm_deberta_proxy"\nTRAIN_VERSION = 170484\nTRAIN_SHA = "40a22ce8d88c5f46feb4637a099db815bef5f713ffe53e89dbe1b7b21de77707"\nTEXT_SHA = "6496071874da8f911fafebd9eec6191e56134a97f00226dd8c78805d31f07e36"\nSTAGE13C_CM_SHA = "c076ee1a507753660bd7356baa3083d277c8fc8e61bedc81bae9192986caad9b"\nSTAGE13C_BA = 95677 / 201150\nCLASSES = (\n    "claude-fable-5", "claude-opus-4-6", "claude-opus-4-8", "claude-opus-5",\n    "claude-sonnet-5", "gemini-3.1-pro-preview", "gemini-3.6-flash",\n    "gemini-3.7-flash", "gemini-3.8-flash", "glm-5.3", "glm-5.3-flash",\n    "gpt-5.5", "gpt-5.6-luna", "gpt-5.6-sol", "gpt-5.6-terra",\n    "gpt-6-astra", "kimi-k3", "qwen3.8-max",\n)\nREPORT_NAME = "assignment_audit_report.json"\nrequire = reference.require\n\n\ndef train_inputs(path):\n    """Recover the canonical fold axes from official bytes without opening a cache."""\n    with path.open("rb") as stream:\n        require(hashlib.file_digest(stream, "sha256").hexdigest() == TRAIN_SHA,\n                "stage13f_official_train_hash")\n        stream.seek(0)\n        frame = pd.read_csv(stream, dtype=str, encoding="utf-8")\n    require(len(frame) == reference.ROWS and set(frame.columns) == {"id", "prompt_id", "model", "text"}\n            and not frame.isna().any().any()\n            and all(frame[c].str.strip().ne("").all() for c in frame.columns), "stage13f_train_schema")\n    ids, prompts, texts = (frame[c].to_numpy(dtype=str) for c in ("id", "prompt_id", "text"))\n    labels = np.asarray(CLASSES, dtype=str)\n    require(reference.digest_json(labels.tolist()) == reference.LABEL_SHA and set(frame.model) == set(CLASSES),\n            "stage13f_train_classes")\n    require(len(np.unique(ids)) == reference.ROWS and len(np.unique(prompts)) == reference.PROMPTS\n            and reference.digest_json(ids.tolist()) == reference.ID_SHA, "stage13f_train_id_axis")\n    require(reference.digest_json([hashlib.sha256(x.encode("utf-8")).hexdigest() for x in texts]) == TEXT_SHA,\n            "stage13f_ordered_text_hash")\n    targets = frame.model.map({label: i for i, label in enumerate(CLASSES)}).to_numpy(dtype=np.int64)\n    require(np.bincount(targets).tolist() == [149] + [150] * 17, "stage13f_train_class_supports")\n    require(not frame.duplicated(["prompt_id", "model"]).any(), "stage13f_duplicate_true_model_in_prompt")\n    folds = np.zeros(reference.ROWS, dtype=np.int64)\n    dummy = np.zeros((reference.ROWS, 1), dtype=np.uint8)\n    for f, (_, selected) in enumerate(GroupKFold(reference.FOLDS).split(dummy, targets, prompts), 1):\n        require((folds[selected] == 0).all(), "stage13f_fold_duplicate_rows")\n        folds[selected] = f\n    masks = prior.fold_masks(prompts, targets, folds)\n    fold_map = {}\n    for group, fold in zip(prompts, folds):\n        require(fold_map.setdefault(str(group), int(fold)) == int(fold), "stage13f_prompt_crosses_folds")\n    require(reference.digest_json(fold_map) == reference.FOLD_SHA, "stage13f_canonical_fold_map_hash")\n    return ids, prompts, texts, targets, folds, labels, masks\n\n\ndef replay_stage13c(glm, deb, y, masks):\n    """Apply unchanged Stage 13C mixing and balancing; verify its frozen confusion."""\n    evidence = np.full((reference.ROWS, reference.K), np.nan, dtype=np.float64)\n    residuals = []\n    for val in masks:\n        mixed = 0.75 * glm[val] + 0.25 * deb[val]\n        evidence[val], residual = reference.evidence(mixed, True)\n        residuals.append(residual)\n    require(np.isfinite(evidence).all(), "stage13f_stage13c_evidence_coverage")\n    control = reference.confusion(y, evidence.argmax(axis=1))\n    digest = reference.digest_json(control.tolist())\n    require(digest == STAGE13C_CM_SHA and abs(float(reference.recalls(control).mean()) - STAGE13C_BA) < 1e-15,\n            "stage13f_stage13c_confusion_replay")\n    return evidence, digest, residuals\n\n\ndef run(input_root, output_root):\n    started = time.monotonic()\n    root, out = Path(input_root), Path(output_root)\n    report = dict(schema_version=1, status=FAILED, scope=SCOPE, fit_attempted=False, fit_performed=False,\n                  cached_probabilities_loaded=False, stage13c_confusion_replayed=False,\n                  paired_comparison_performed=False, full_s07_oof_admitted=False,\n                  official_score_estimated=False, official_submission_created=False,\n                  competition_test_read=False, raw_rows_or_predictions_exported=False,\n                  uses_reused_development_prompts=True, s07_typography_guard_used=False,\n                  train_sha256=TRAIN_SHA, train_ordered_text_sha256=TEXT_SHA,\n                  canonical_prompt_fold_mapping_sha256=reference.FOLD_SHA,\n                  expected_input_version_ids=[TRAIN_VERSION, reference.SOURCE_VERSION, prior.DEBERTA_VERSION],\n                  expected_heldout_cache_sha256=reference.CACHE_SHA,\n                  expected_deberta_cache_sha256=prior.DEBERTA_SHA,\n                  sklearn_version=sklearn_version, numpy_version=np.__version__, scipy_version=scipy_version)\n    writable = False\n    try:\n        require(not out.exists() or (out.is_dir() and not any(out.iterdir())), "fresh_output_directory")\n        out.mkdir(parents=True, exist_ok=True)\n        writable = True\n        require(sklearn_version == prior.SKLEARN_VERSION, "stage13f_sklearn_version")\n        ids, prompts, texts, y, folds, classes, masks = train_inputs(reference.one(root, "train.csv"))\n        with threadpool_limits(limits=prior.THREADS):\n            geometry, cluster_ids = assignment.geometry(texts, folds, prompts, y)\n        eligible_rows = geometry["eligible_rows"]\n        eligible_clusters = geometry["eligible_clusters"]\n        require(type(eligible_rows) is int and type(eligible_clusters) is int and\n                0 <= eligible_clusters * 12 <= eligible_rows <= min(eligible_clusters * 18, reference.ROWS),\n                "stage13f_geometry_coverage")\n        require(len(geometry["folds"]) == reference.FOLDS and cluster_ids.shape == folds.shape\n                and cluster_ids.dtype.kind in "iu", "stage13f_geometry_folds_or_cluster_axis")\n        report.update(geometry=geometry, rows=reference.ROWS, prompts=reference.PROMPTS, classes=classes.tolist())\n        if eligible_rows == 0:\n            report["status"] = COMPLETE_ZERO\n        else:\n            saved_ids, saved_prompts, saved_y, saved_folds, saved_classes, glm, producer = reference.cache_inputs(root)\n            require(all(np.array_equal(a, b) for a, b in zip((ids, prompts, y, folds, classes),\n                                                            (saved_ids, saved_prompts, saved_y, saved_folds, saved_classes))),\n                    "stage13f_heldout_cache_row_or_fold_alignment")\n            report["cached_probabilities_loaded"] = True\n            prior.replay_control(glm, y, masks, producer)\n            report["glm_controls_replayed"] = True\n            features, id_mode = prior.load_features(reference.one(root, "enc_debv3l2_train.npz"), ids)\n            report["fit_attempted"] = True\n            deb, fit_details, pools = prior.fit_oof(features, y, masks)\n            report.update(fit_performed=True, deberta_fit_folds=fit_details, thread_pools=pools,\n                          deberta_id_verification=id_mode, feature_layers=[24, 6])\n            evidence, digest, residuals = replay_stage13c(glm, deb, y, masks)\n            report.update(stage13c_confusion_replayed=True, stage13c_confusion_sha256=digest,\n                          mixture_weights={"glm": 0.75, "deberta": 0.25},\n                          balancing={"alpha": 1, "iterations": 100, "fold_local": True,\n                                     "maximum_column_mass_errors": residuals})\n            evidence_sha = hashlib.sha256(evidence.astype("<f8", copy=False).tobytes()).hexdigest()\n            comparison = assignment.compare(evidence, cluster_ids, folds, prompts, y)\n            require(comparison["eligible_rows"] == eligible_rows\n                    and comparison["eligible_clusters"] == eligible_clusters\n                    and comparison["assignment_off_confusion_sha256"] == digest,\n                    "stage13f_comparison_coverage_or_control_replay")\n            require(hashlib.sha256(evidence.astype("<f8", copy=False).tobytes()).hexdigest() == evidence_sha,\n                    "stage13f_evidence_mutated")\n            require(hashlib.sha256(glm.astype("<f8", copy=False).tobytes()).hexdigest() == reference.PROB_SHA,\n                    "stage13f_glm_cache_mutated")\n            report.update(status=COMPLETE_PAIRED, paired_comparison_performed=True, comparison=comparison)\n    except reference.GateError as error:\n        report.update(failure_code=str(error))\n    except Exception as error:\n        report.update(failure_code="runtime_failure", failure_type=type(error).__name__)\n    report["seconds"] = round(time.monotonic() - started, 6)\n    if writable:\n        try:\n            saved = out / REPORT_NAME\n            tmp = out / ("." + REPORT_NAME + ".tmp")\n            tmp.write_text(json.dumps(report, sort_keys=True, indent=2, allow_nan=False) + "\\n", encoding="utf-8")\n            tmp.replace(saved)\n            require(json.loads(saved.read_text(encoding="utf-8")) == report, "stage13f_saved_report_mismatch")\n        except Exception as error:\n            report.update(status=FAILED, failure_code="report_write_or_readback_failure", failure_type=type(error).__name__)\n    return report\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--input-root", default="/kaggle/input")\n    parser.add_argument("--output-root", default="/kaggle/working/stage13f_assignment_audit")\n    args = parser.parse_args()\n    result = run(args.input_root, args.output_root)\n    print("STAGE13F_ASSIGNMENT_AUDIT_JSON " + json.dumps(result, sort_keys=True, allow_nan=False), flush=True)\n    return 0 if result["status"] in (COMPLETE_ZERO, COMPLETE_PAIRED) else 2\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'be418f7603ec61a1920061bd063494eb17d609ad5c64495b3c404672adf0f720'), 'stage13f_assignment_core.py': ('"""Isolate S07\'s dense-group assignment on a held-out development proxy."""\nfrom __future__ import annotations\n\nfrom collections import Counter\nfrom fractions import Fraction\nimport hashlib\n\nimport numpy as np\nfrom scipy.optimize import linear_sum_assignment\nfrom sklearn.feature_extraction.text import TfidfVectorizer\n\nimport stage13b_balancing_stress as reference\n\n\nK = reference.K\nTHRESHOLD, CAP, MIN_GROUP = 0.15, 18, 12\nTFIDF = dict(sublinear_tf=True, stop_words="english", min_df=2, max_df=0.05)\nrequire = reference.require\n\n\ndef capped_agglo(S, cap=18, thr=0.02):\n    """S07\'s float32 average linkage, including its row-order tie policy."""\n    n = len(S); A = S.astype(np.float32).copy(); np.fill_diagonal(A, -np.inf)\n    size = np.ones(n, np.int32); alive = np.ones(n, bool); lab = np.arange(n)\n    while True:\n        ok = (size[:, None] + size[None, :] <= cap) & alive[:, None] & alive[None, :]\n        M = np.where(ok, A, -np.inf); k = int(np.argmax(M)); i, j = divmod(k, n)\n        if not np.isfinite(M[i, j]) or M[i, j] < thr: break\n        new = (size[i] * A[i] + size[j] * A[j]) / (size[i] + size[j])\n        A[i] = new; A[:, i] = new; A[i, i] = -np.inf; A[j] = -np.inf; A[:, j] = -np.inf\n        size[i] += size[j]; alive[j] = False; lab[lab == j] = i\n    return lab\n\n\ndef infer_groups(texts):\n    """One scoring batch only; reproduce the source\'s singleton fallback."""\n    try:\n        features = TfidfVectorizer(**TFIDF).fit_transform(texts)\n        clusters = capped_agglo((features @ features.T).toarray(), CAP, THRESHOLD)\n    except ValueError:\n        return np.arange(len(texts), dtype=np.int64), True\n    return clusters, False\n\n\ndef validate_folds(folds):\n    require(folds.ndim == 1 and len(folds) > 0 and folds.dtype.kind in "iu"\n            and (folds > 0).all(), "stage13f_fold_axis")\n\n\ndef validate_truth(folds, groups, targets):\n    require(groups.shape == targets.shape == folds.shape and groups.dtype.kind == "U"\n            and np.char.str_len(groups).min() > 0 and targets.dtype.kind in "iu"\n            and ((targets >= 0) & (targets < K)).all(), "stage13f_truth_axes")\n    assigned = {}\n    for prompt, fold in zip(groups, folds, strict=True):\n        require(assigned.setdefault(prompt, fold) == fold, "stage13f_prompt_crosses_folds")\n\n\ndef buckets(clusters):\n    mapping = {}\n    for index, label in enumerate(clusters):\n        mapping.setdefault(int(label), []).append(index)\n    return [np.asarray(mapping[label], dtype=np.int64) for label in sorted(mapping)]\n\n\ndef eligible(cluster_buckets):\n    return [rows for rows in cluster_buckets if MIN_GROUP <= len(rows) <= CAP]\n\n\ndef group_statistics(cluster_buckets, groups, targets):\n    selected = eligible(cluster_buckets)\n    rows = sum(map(len, selected))\n    prompt_counters = [Counter(groups[idx]) for idx in selected]\n    majority_rows = sum(max(c.values()) for c in prompt_counters)\n    duplicate_counts = [len(idx) - len(np.unique(targets[idx])) for idx in selected]\n    all_sizes = Counter(map(len, cluster_buckets))\n    require(not all_sizes or max(all_sizes) <= CAP, "stage13f_cluster_size_cap")\n    return dict(inferred_clusters=len(cluster_buckets),\n                cluster_size_histogram={str(s): all_sizes[s] for s in sorted(all_sizes)},\n                eligible_clusters=len(selected), eligible_rows=rows,\n                eligible_mixed_prompt_clusters=sum(len(c) > 1 for c in prompt_counters),\n                eligible_clusters_with_duplicate_true_label=sum(d > 0 for d in duplicate_counts),\n                eligible_duplicate_true_label_excess_rows=sum(duplicate_counts),\n                eligible_majority_prompt_rows=majority_rows,\n                eligible_row_weighted_prompt_purity=(majority_rows / rows if rows else None))\n\n\ndef geometry(texts, folds, groups, targets):\n    """Cluster text independently in each fold; only then inspect truth."""\n    validate_folds(folds)\n    require(texts.shape == folds.shape and texts.dtype.kind == "U", "stage13f_text_axis")\n    cluster_ids = np.full(len(folds), -1, dtype=np.int64)\n    entries = []\n    offset = 0\n    for fold in np.unique(folds):\n        index = np.flatnonzero(folds == fold)\n        local, fallback = infer_groups(texts[index])\n        require(local.shape == index.shape and local.dtype.kind in "iu"\n                and ((0 <= local) & (local < len(index))).all(), "stage13f_inferred_axis")\n        cluster_ids[index] = local + offset\n        offset += len(index)\n        entries.append((int(fold), index, local, fallback))\n    require(offset == len(folds) and (cluster_ids >= 0).all(), "stage13f_cluster_coverage")\n    validate_truth(folds, groups, targets)\n    reports = [dict(fold=fold, rows=len(index), true_prompts=len(np.unique(groups[index])),\n                    singleton_fallback=fallback, **group_statistics(buckets(local), groups[index], targets[index]))\n               for fold, index, local, fallback in entries]\n    pooled = group_statistics(buckets(cluster_ids), groups, targets)\n    require(all(pooled[key] == sum(r[key] for r in reports) for key in\n                ("inferred_clusters", "eligible_clusters", "eligible_rows", "eligible_majority_prompt_rows")),\n            "stage13f_cluster_pool_reconciliation")\n    return (dict(**pooled, folds=reports, tfidf=TFIDF.copy(), cluster_cap=CAP,\n                 average_cosine_cut=THRESHOLD, assignment_size_min=MIN_GROUP,\n                 assignment_size_max=CAP, grouping_uses_only_batch_text=True), cluster_ids)\n\n\ndef paired_counts(targets, off, on):\n    co, cn = off == targets, on == targets\n    fixed, broken = int((cn & ~co).sum()), int((co & ~cn).sum())\n    changed = int((on != off).sum())\n    require(fixed + broken <= changed and fixed - broken == int(cn.sum() - co.sum()), "stage13f_paired_counts")\n    return dict(corrected=fixed, broken=broken, changed=changed)\n\n\ndef compare(balanced_log_evidence, cluster_ids, folds, groups, targets):\n    """Toggle just the source\'s assignment on the same fixed log evidence."""\n    validate_folds(folds)\n    validate_truth(folds, groups, targets)\n    require(cluster_ids.shape == folds.shape and cluster_ids.dtype.kind in "iu"\n            and (cluster_ids >= 0).all(), "stage13f_cluster_axis")\n    require(balanced_log_evidence.shape == (len(folds), K) and balanced_log_evidence.dtype == np.float64\n            and np.isfinite(balanced_log_evidence).all(), "stage13f_log_evidence_axis")\n    require(np.max(np.abs(np.exp(balanced_log_evidence).sum(axis=1) - (1 + K * 1e-12))) <= 2e-12,\n            "stage13f_log_evidence_normalization")\n    before = hashlib.sha256(balanced_log_evidence.astype("<f8", copy=False).tobytes()).hexdigest()\n    inferred = buckets(cluster_ids)\n    require(max(map(len, inferred)) <= CAP and all(np.unique(folds[idx]).size == 1 for idx in inferred),\n            "stage13f_assignment_cluster_boundaries")\n    dense = eligible(inferred)\n    require(dense, "stage13f_no_assignment_eligible_groups")\n    off = balanced_log_evidence.argmax(axis=1)\n    on = off.copy()\n    covered = np.zeros(len(folds), dtype=bool)\n    for idx in dense:\n        r, c = linear_sum_assignment(-balanced_log_evidence[idx])\n        require(np.array_equal(r, np.arange(len(idx))) and len(np.unique(c)) == len(idx) and not covered[idx].any(),\n                "stage13f_assignment_unique_coverage")\n        on[idx[r]], covered[idx] = c, True\n    require(np.array_equal(off[~covered], on[~covered])\n            and hashlib.sha256(balanced_log_evidence.astype("<f8", copy=False).tobytes()).hexdigest() == before,\n            "stage13f_assignment_changed_fixed_evidence_or_outside_cluster")\n    targets = targets.astype(np.int64, copy=False)\n    cm0, cm1 = reference.confusion(targets, off), reference.confusion(targets, on)\n    recall0, recall1 = reference.recalls(cm0), reference.recalls(cm1)\n    supports = cm0.sum(axis=1)\n    deltas = [Fraction(int(cm1[i, i]) - int(cm0[i, i]), int(supports[i])) * 100 for i in range(K)]\n    delta = sum(deltas) / K\n    fold_reports = []\n    rng = np.random.default_rng(reference.SEED)\n    support_draws = np.zeros((reference.DRAWS, K), dtype=np.int64)\n    diff_draws = np.zeros_like(support_draws)\n    for fold in np.unique(folds):\n        mask = folds == fold\n        yt, pf, qf = targets[mask], off[mask], on[mask]\n        c0, c1 = reference.confusion(yt, pf), reference.confusion(yt, qf)\n        groups_f = groups[mask]\n        gp, idx = np.unique(groups_f, return_inverse=True)\n        prompt_support = np.zeros((len(gp), K), dtype=np.int64)\n        prompt_diff = np.zeros_like(prompt_support)\n        np.add.at(prompt_support, (idx, yt), 1)\n        np.add.at(prompt_diff, (idx, yt), (qf == yt).astype(np.int64) - (pf == yt).astype(np.int64))\n        w = rng.multinomial(len(gp), np.full(len(gp), 1 / len(gp)), size=reference.DRAWS)\n        support_draws += w @ prompt_support\n        diff_draws += w @ prompt_diff\n        off_ba, on_ba = float(reference.recalls(c0).mean()), float(reference.recalls(c1).mean())\n        fold_reports.append(dict(fold=int(fold), rows=int(mask.sum()), eligible_rows=int((covered & mask).sum()),\n                                 assignment_off_ba=off_ba, assignment_on_ba=on_ba,\n                                 delta_pp=100 * (on_ba - off_ba), **paired_counts(yt, pf, qf)))\n    require((support_draws > 0).all(), "stage13f_bootstrap_missing_class")\n    counts = paired_counts(targets, off, on)\n    require(all(counts[key] == sum(f[key] for f in fold_reports) for key in counts)\n            and counts["changed"] <= int(covered.sum())\n            and np.array_equal(cm0.sum(1), cm1.sum(1)), "stage13f_assignment_pair_reconciliation")\n    interval = np.percentile((diff_draws / support_draws).mean(axis=1) * 100, [2.5, 97.5]).tolist()\n    return dict(assignment_off_ba=float(recall0.mean()), assignment_on_ba=float(recall1.mean()),\n                delta_pp=float(delta), delta_pp_exact=str(delta), conditional_delta_ci95_pp=interval,\n                class_supports=supports.tolist(), class_delta_pp=[float(d) for d in deltas],\n                assignment_off_confusion_counts=cm0.tolist(), assignment_on_confusion_counts=cm1.tolist(),\n                assignment_off_confusion_sha256=reference.digest_json(cm0.tolist()),\n                assignment_on_confusion_sha256=reference.digest_json(cm1.tolist()),\n                eligible_clusters=len(dense), eligible_rows=int(covered.sum()), folds=fold_reports,\n                outside_eligible_clusters_unchanged=True, fixed_log_evidence_unchanged=True,\n                bootstrap=dict(draws=reference.DRAWS, seed=reference.SEED,\n                               scope="conditional true-prompt resampling on fixed scores, clusters and decisions",\n                               limitation="inferred groups may couple true prompts; not pipeline-level uncertainty"),\n                **counts)\n', '9d4a64db8fb3f2853234c10d98617f0219184d3269b54e0bba87b9883bdc8e8c')}
source_dir = Path("/kaggle/working/_stage13i_sources")
source_dir.mkdir(parents=True, exist_ok=True)
for name, (code, sha) in SOURCES.items():
    data = code.encode("utf-8")
    path = source_dir / name
    if hashlib.sha256(data).hexdigest() != sha:
        raise RuntimeError("Stage 13I source checksum mismatch")
    path.write_bytes(data)
    if hashlib.sha256(path.read_bytes()).hexdigest() != sha:
        raise RuntimeError("Stage 13I written source checksum mismatch")
print("STAGE13I_SOURCE_CHECKS_PASS", SOURCES['stage13i_gemini_audit.py'][1], flush=True)
subprocess.run([sys.executable, "-B", str(source_dir / 'stage13i_gemini_audit.py'),
                "--input-root", "/kaggle/input", "--output-root", "/kaggle/working/stage13i_gemini_audit"],
               check=True, timeout=900)
